# 0DTE SPX Options Analysis

**Data Source**: Local parquet files (from Polygon flat-file backfill)  
**Method**: Side-Weighted GEX with vectorized Black-Scholes  
**Charts**: Net Drift, Net Flow, Combined GEX+DEX, Vomma/Zomma, CAR, Volatility Skew, Trade Distribution

## Quick Start
1. Set `TRADE_DATE` below to any available date (2024-01-02 through 2025-03-20)
2. Run all cells
3. Charts render inline with dark theme

In [ ]:
# Setup & Configuration
import sys
from pathlib import Path

# Add project root to path for src imports
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd

from src.config import AnalysisConfig
from src.data_loader import GEXDataLoader
from src.gex_calculator import calculate_gex
from src.black_scholes import calculate_greeks

# ========== CONFIGURATION ==========
TRADE_DATE = "2024-06-14"   # Any date from data/ directory (2024-01-02 to 2025-03-20)
CUTOFF_TIME = None           # None for full day, or "14:00" for partial day
DARK_THEME = True
GEX_REPRESENTATION = "pct"   # "pct" = per 1% move, "dollar" = per $1 move
MONEYNESS_FILTER = "otm"    # "all", "otm", "atm", "itm"
ATM_TOLERANCE = 10           # Strikes within +/-$10 of spot = ATM
# ===================================

ET = ZoneInfo("America/New_York")
DATA_DIR = PROJECT_ROOT / "data"
analysis_config = AnalysisConfig(trade_date=TRADE_DATE)

CUTOFF_TS = None
TIME_LABEL = "Full Day"
if CUTOFF_TIME:
    h, m = map(int, CUTOFF_TIME.split(':'))
    y, mo, d = map(int, TRADE_DATE.split('-'))
    CUTOFF_TS = datetime(y, mo, d, h, m, tzinfo=ET)
    TIME_LABEL = f"as of {CUTOFF_TIME} ET"

REP_LABELS = {"pct": "Per 1% Move", "dollar": "Per $1 Move", "unit": "Per 1 Unit"}
REP_LABEL = REP_LABELS.get(GEX_REPRESENTATION, "Per 1% Move")
MONEYNESS_LABELS = {
    "all": "All Options", "otm": "OTM Only",
    "atm": f"ATM Only (+/-${ATM_TOLERANCE})", "itm": "ITM Only"
}
MONEYNESS_LABEL = MONEYNESS_LABELS.get(MONEYNESS_FILTER, "All Options")
GENERATED_AT = datetime.now(ET).strftime('%Y-%m-%d %H:%M:%S ET')

# List available dates
available = sorted([f.stem.replace('trades_', '') for f in DATA_DIR.glob('trades_*.parquet')])
print(f"Available dates: {len(available)} ({available[0]} to {available[-1]})")
print(f"Config: {TRADE_DATE} | {TIME_LABEL}")
print(f"Representation: {REP_LABEL} | Moneyness: {MONEYNESS_LABEL}")
print(f"Generated: {GENERATED_AT}")

In [ ]:
# Load Data from Local Parquet Files
loader = GEXDataLoader(analysis_config, DATA_DIR)
df = loader.load(verbose=True)

if df.empty:
    raise ValueError(f"No data found for {TRADE_DATE}. Check data/ directory.")

if CUTOFF_TS:
    df = df[df['timestamp'] <= CUTOFF_TS].copy()
    print(f"After cutoff: {len(df):,} trades")

print(f"Loaded: {len(df):,} trades | {df['size'].sum():,} contracts")

# =============================================================================
# MONEYNESS FILTER
# =============================================================================
def apply_moneyness_filter(data, filter_type, atm_tol=10):
    """Filter trades by moneyness relative to spot price."""
    if filter_type == "all":
        return data
    is_call = data['opt_type'] == 'C'
    is_put = data['opt_type'] == 'P'
    spot = data['spot']
    strike = data['strike']
    if filter_type == "otm":
        mask = (is_call & (strike > spot)) | (is_put & (strike < spot))
    elif filter_type == "atm":
        mask = abs(strike - spot) <= atm_tol
    elif filter_type == "itm":
        mask = (is_call & (strike < spot)) | (is_put & (strike > spot))
    else:
        return data
    return data[mask].copy()

df_original_count = len(df)
df = apply_moneyness_filter(df, MONEYNESS_FILTER, ATM_TOLERANCE)
print(f"Moneyness filter ({MONEYNESS_LABEL}): {len(df):,} trades ({len(df)/df_original_count*100:.1f}% of total)")

# Calculate GEX
result = calculate_gex(df, analysis_config)

print(f"\nTrade direction:")
print(df['trade_dir'].value_counts())
print(f"\nSide breakdown:")
print(df['side'].value_counts())

# Calculate premium
df['premium'] = df['price'] * df['size'] * 100

# Aggregate to minute-level data
df['minute'] = df['timestamp'].dt.floor('1min')

# =============================================================================
# NET DRIFT DATA: EXCLUDE mid_market trades
# =============================================================================
df_no_mid = df[df['side'] != 'mid_market'].copy()
print(f"\nNet Drift: Using {len(df_no_mid):,} trades (excluding {len(df) - len(df_no_mid):,} mid_market)")

drift_agg = df_no_mid.groupby(['minute', 'opt_type', 'trade_dir']).agg({
    'premium': 'sum',
    'size': 'sum',
    'spot': 'last'
}).reset_index()

mid_market_agg = df[df['side'] == 'mid_market'].groupby(['minute', 'opt_type']).agg({
    'premium': 'sum',
    'size': 'sum'
}).reset_index()

# Pivot to get call_buy, call_sell, put_buy, put_sell columns
drift_data = []
for minute in sorted(df['minute'].unique()):
    row = {'minute': minute}
    minute_df = drift_agg[drift_agg['minute'] == minute]
    mid_minute_df = mid_market_agg[mid_market_agg['minute'] == minute]
    
    for opt, dir_name, prem_col, vol_col in [
        ('C', 'BUY', 'call_buy_premium', 'call_buy_volume'),
        ('C', 'SELL', 'call_sell_premium', 'call_sell_volume'),
        ('P', 'BUY', 'put_buy_premium', 'put_buy_volume'),
        ('P', 'SELL', 'put_sell_premium', 'put_sell_volume'),
    ]:
        match = minute_df[(minute_df['opt_type'] == opt) & (minute_df['trade_dir'] == dir_name)]
        row[prem_col] = match['premium'].sum() if len(match) > 0 else 0
        row[vol_col] = match['size'].sum() if len(match) > 0 else 0
    
    for opt, prem_col, vol_col in [
        ('C', 'call_mid_premium', 'call_mid_volume'),
        ('P', 'put_mid_premium', 'put_mid_volume'),
    ]:
        match = mid_minute_df[mid_minute_df['opt_type'] == opt]
        row[prem_col] = match['premium'].sum() if len(match) > 0 else 0
        row[vol_col] = match['size'].sum() if len(match) > 0 else 0
    
    row['avg_spot'] = minute_df['spot'].iloc[-1] if len(minute_df) > 0 else None
    drift_data.append(row)

df_drift = pd.DataFrame(drift_data).sort_values('minute')
df_drift['net_call'] = df_drift['call_buy_premium'] - df_drift['call_sell_premium']
df_drift['net_put'] = df_drift['put_buy_premium'] - df_drift['put_sell_premium']
df_drift['cum_call'] = df_drift['net_call'].cumsum()
df_drift['cum_put'] = df_drift['net_put'].cumsum()
df_drift['cum_total'] = df_drift['cum_call'] + df_drift['cum_put']

# FLOW data (aggressive buys only - at_ask + above_ask)
df_aggressive = df[df['side'].isin(['at_ask', 'above_ask'])].copy()
print(f"Net Flow: Using {len(df_aggressive):,} aggressive buy trades")

flow_agg = df_aggressive.groupby(['minute', 'opt_type']).agg({
    'premium': 'sum',
    'size': 'sum',
    'spot': 'last'
}).reset_index()

flow_data = []
for minute in sorted(df['minute'].unique()):
    row = {'minute': minute}
    minute_flow = flow_agg[flow_agg['minute'] == minute]
    
    for opt, prem_col, vol_col in [
        ('C', 'call_flow_premium', 'call_flow_volume'),
        ('P', 'put_flow_premium', 'put_flow_volume'),
    ]:
        match = minute_flow[minute_flow['opt_type'] == opt]
        row[prem_col] = match['premium'].sum() if len(match) > 0 else 0
        row[vol_col] = match['size'].sum() if len(match) > 0 else 0
    
    row['spot'] = minute_flow['spot'].iloc[-1] if len(minute_flow) > 0 else None
    flow_data.append(row)

df_flow = pd.DataFrame(flow_data).sort_values('minute')
df_flow['cum_call_flow'] = df_flow['call_flow_premium'].cumsum()
df_flow['cum_put_flow'] = df_flow['put_flow_premium'].cumsum()
df_flow['cum_total_flow'] = df_flow['cum_call_flow'] + df_flow['cum_put_flow']

print(f"\nData prepared. Ready for charts.")

In [ ]:
# ============================================================
# 2. NET DRIFT CHART (Excluding Mid-Market, like QuantData)
# ============================================================
# Net Drift shows cumulative premium by sentiment:
# - Call line: Ask-side premium - Bid-side premium
# - Put line: Ask-side premium - Bid-side premium  
# Mid-market trades are EXCLUDED from the calculation

if drift_data:
    times = []
    # Call and Put net drift (ask - bid, excluding mid_market)
    call_net_drift = []
    put_net_drift = []
    spot_prices = []
    vol_minute = []
    vol_cumulative = []

    cumulative_call = cumulative_put = 0.0
    cumulative_vol = 0

    for row in drift_data:
        # Convert to ET for display
        ts = row["minute"]
        if ts.tzinfo is None:
            ts = ts.tz_localize('UTC')
        times.append(ts.tz_convert(ET))

        # Net drift = Ask-side - Bid-side (excluding mid_market)
        call_ask = float(row.get("call_buy_premium") or 0)   # at_ask, above_ask
        call_bid = float(row.get("call_sell_premium") or 0)  # at_bid, below_bid
        put_ask = float(row.get("put_buy_premium") or 0)
        put_bid = float(row.get("put_sell_premium") or 0)

        call_net = call_ask - call_bid  # Positive = more call buying
        put_net = put_ask - put_bid     # Positive = more put buying (bearish)

        cumulative_call += call_net
        cumulative_put += put_net

        call_net_drift.append(cumulative_call)
        put_net_drift.append(cumulative_put)

        spot_prices.append(float(row["avg_spot"]) if row["avg_spot"] else None)

        # Volume (excluding mid_market)
        call_buy_vol = int(row.get("call_buy_volume") or 0)
        call_sell_vol = int(row.get("call_sell_volume") or 0)
        put_buy_vol = int(row.get("put_buy_volume") or 0)
        put_sell_vol = int(row.get("put_sell_volume") or 0)

        net_vol = (call_buy_vol - call_sell_vol) - (put_buy_vol - put_sell_vol)
        vol_minute.append(net_vol)
        cumulative_vol += net_vol
        vol_cumulative.append(cumulative_vol)

    # Determine scale
    all_values = call_net_drift + put_net_drift
    max_drift = max(abs(max(all_values)), abs(min(all_values))) if all_values else 1
    scale = 1_000_000 if max_drift > 500_000 else 1_000
    scale_label = "M" if scale == 1_000_000 else "K"

    call_scaled = np.array(call_net_drift) / scale
    put_scaled = np.array(put_net_drift) / scale
    vol_minute_k = np.array(vol_minute) / 1000
    vol_cumulative_k = np.array(vol_cumulative) / 1000

    # Call color: green if positive, red if negative
    call_color = "#22c55e" if cumulative_call >= 0 else "#ef4444"
    # Put color: always red (puts represent bearish sentiment)
    put_color = "#ef4444"

    # Create figure with 3 panels (increased spacing)
    if DARK_THEME:
        plt.style.use("dark_background")
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 13), height_ratios=[3, 1, 1], gridspec_kw={"hspace": 0.28})

    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2, ax3]:
            ax.set_facecolor("#0a1628")

    # Main chart - Net Drift (solid lines)
    ax1.plot(times, call_scaled, color=call_color, linewidth=2, label=f"Calls (${call_scaled[-1]:.1f}{scale_label})")
    ax1.plot(times, put_scaled, color=put_color, linewidth=2, label=f"Puts (${put_scaled[-1]:.1f}{scale_label})")

    all_drift = np.concatenate([call_scaled, put_scaled])
    y_min = min(all_drift.min(), 0) * 1.3
    y_max = max(all_drift.max(), 0) * 1.3
    if y_max - y_min < 1:
        y_min, y_max = -10, 10
    ax1.set_ylim(y_min, y_max)

    # Spot price overlay
    ax1_price = ax1.twinx()
    valid_spots = [(t, s) for t, s in zip(times, spot_prices) if s]
    if valid_spots:
        st, sv = zip(*valid_spots)
        ax1_price.plot(st, sv, color="white", linewidth=1.5, alpha=0.7, label=f"Spot (${sv[-1]:.2f})")
        ax1_price.set_ylabel("Underlying ($)", fontsize=11, color="white")
        ax1_price.tick_params(axis="y", colors="white")

    ax1.axhline(y=0, color="#4a5568", linestyle="-", linewidth=0.8, alpha=0.5)
    ax1.set_ylabel(f"Cumulative Premium (${scale_label})", fontsize=12)
    ax1.set_title(f"Net Drift — {TRADE_DATE} | {TIME_LABEL}\n"
                  f"Ask-Side Premium - Bid-Side Premium (Mid-Market Excluded)",
                  fontsize=14, color="white", pad=15, fontweight="bold")
    ax1.tick_params(colors="white")
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax1.grid(axis="both", alpha=0.2, color="#4a5568")

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_price.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", ncol=3, fontsize=9, 
               framealpha=0.8, facecolor="#1a2744" if DARK_THEME else "white")

    # Panel 2: Per-minute net volume (bar chart)
    colors_minute = ["#22c55e" if v >= 0 else "#ef4444" for v in vol_minute_k]
    ax2.bar(times, vol_minute_k, width=0.0005, color=colors_minute, alpha=0.8)
    ax2.axhline(y=0, color="#4a5568", linestyle="-", linewidth=0.5)
    ax2.set_ylabel("Net Vol/min (K)", fontsize=10)
    ax2.set_title("Per-Minute Net Volume (Call Net - Put Net)", fontsize=11, color="white", pad=8)
    ax2.tick_params(colors="white", labelsize=9)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax2.grid(axis="both", alpha=0.2, color="#4a5568")
    net_vol_total = vol_cumulative_k[-1] if len(vol_cumulative_k) > 0 else 0
    ax2.text(0.02, 0.90, f"Total Net: {net_vol_total:+,.0f}K contracts", transform=ax2.transAxes, fontsize=10,
             verticalalignment="top", bbox=dict(boxstyle="round", facecolor="#1a2744" if DARK_THEME else "white", alpha=0.8))

    # Panel 3: Cumulative volume (bar chart like Net vol/min)
    colors_cumulative = ["#22c55e" if v >= 0 else "#ef4444" for v in vol_cumulative_k]
    ax3.bar(times, vol_cumulative_k, width=0.0005, color=colors_cumulative, alpha=0.8)
    ax3.axhline(y=0, color="#4a5568", linestyle="-", linewidth=0.5)
    ax3.set_ylabel("Cumulative (K)", fontsize=10)
    ax3.set_xlabel("Time (ET)", fontsize=11)
    ax3.set_title("Cumulative Net Volume", fontsize=11, color="white", pad=8)
    ax3.tick_params(colors="white", labelsize=9)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax3.grid(axis="both", alpha=0.2, color="#4a5568")
    direction = "BULLISH" if vol_cumulative_k[-1] >= 0 else "BEARISH"
    direction_color = "#22c55e" if vol_cumulative_k[-1] >= 0 else "#ef4444"
    ax3.text(0.02, 0.90, f"Direction: {direction} ({vol_cumulative_k[-1]:+,.0f}K)", transform=ax3.transAxes, fontsize=10,
             verticalalignment="top", color=direction_color,
             bbox=dict(boxstyle="round", facecolor="#1a2744" if DARK_THEME else "white", alpha=0.8))

    for ax in [ax1, ax2, ax3]:
        for spine in ax.spines.values():
            spine.set_color("#4a5568")

    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom", fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout()
    plt.show()
    
    # Summary
    print(f"\nNet Drift Summary (Mid-Market Excluded):")
    print(f"  Call Net: ${cumulative_call:,.0f} ({'BULLISH' if cumulative_call > 0 else 'BEARISH'})")
    print(f"  Put Net:  ${cumulative_put:,.0f} ({'BEARISH' if cumulative_put > 0 else 'BULLISH'})")
else:
    print("No drift data available")

In [ ]:
# ============================================================
# 3. NET FLOW CHART (Aggressive Buys Only - at_ask + above_ask)
# ============================================================
# Net Flow shows raw buying pressure from aggressive buyers only
# This is different from Net Drift which shows net of all trades
# - Only includes trades executed at or above the ask price
# - No subtraction - just cumulative premium of aggressive buys

# Filter to aggressive buys only (at_ask, above_ask)
df_aggressive = df[df['side'].isin(['at_ask', 'above_ask'])].copy()
print(f"Net Flow: Using {len(df_aggressive):,} aggressive buy trades (at_ask + above_ask)")

if len(df_aggressive) > 0:
    # Aggregate by minute and option type
    flow_agg = df_aggressive.groupby(['minute', 'opt_type']).agg({
        'premium': 'sum',
        'size': 'sum',
        'spot': 'last'
    }).reset_index()
    
    # Build flow data
    flow_data = []
    for minute in sorted(df_aggressive['minute'].unique()):
        row = {'minute': minute}
        minute_df = flow_agg[flow_agg['minute'] == minute]
        
        call_match = minute_df[minute_df['opt_type'] == 'C']
        put_match = minute_df[minute_df['opt_type'] == 'P']
        
        row['call_flow_premium'] = call_match['premium'].sum() if len(call_match) > 0 else 0
        row['put_flow_premium'] = put_match['premium'].sum() if len(put_match) > 0 else 0
        row['call_flow_volume'] = call_match['size'].sum() if len(call_match) > 0 else 0
        row['put_flow_volume'] = put_match['size'].sum() if len(put_match) > 0 else 0
        row['avg_spot'] = minute_df['spot'].iloc[0] if len(minute_df) > 0 else None
        
        flow_data.append(row)
    
    # Process cumulative flow
    times = []
    cumulative_call_flow = []
    cumulative_put_flow = []
    spot_prices = []
    call_vol_minute = []
    put_vol_minute = []
    
    cum_call = cum_put = 0.0
    
    for row in flow_data:
        # Convert to ET for display
        ts = row['minute']
        if ts.tzinfo is None:
            ts = ts.tz_localize('UTC')
        times.append(ts.tz_convert(ET))
        
        call_prem = float(row.get('call_flow_premium') or 0)
        put_prem = float(row.get('put_flow_premium') or 0)
        
        cum_call += call_prem
        cum_put += put_prem
        
        cumulative_call_flow.append(cum_call)
        cumulative_put_flow.append(cum_put)
        spot_prices.append(float(row['avg_spot']) if row.get('avg_spot') else None)
        call_vol_minute.append(int(row.get('call_flow_volume') or 0))
        put_vol_minute.append(int(row.get('put_flow_volume') or 0))
    
    # Determine scale
    max_flow = max(cum_call, cum_put, 1)
    scale = 1_000_000 if max_flow > 500_000 else 1_000
    scale_label = "M" if scale == 1_000_000 else "K"
    
    call_scaled = np.array(cumulative_call_flow) / scale
    put_scaled = np.array(cumulative_put_flow) / scale
    
    # Create figure with 2 panels
    if DARK_THEME:
        plt.style.use("dark_background")
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), height_ratios=[3, 1], gridspec_kw={"hspace": 0.12})
    
    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2]:
            ax.set_facecolor("#0a1628")
    
    # Main chart - Cumulative Flow (filled areas)
    ax1.fill_between(times, 0, call_scaled, color="#22c55e", alpha=0.3, label=f"Call Flow (${call_scaled[-1]:.1f}{scale_label})")
    ax1.fill_between(times, 0, put_scaled, color="#ef4444", alpha=0.3, label=f"Put Flow (${put_scaled[-1]:.1f}{scale_label})")
    ax1.plot(times, call_scaled, color="#22c55e", linewidth=2)
    ax1.plot(times, put_scaled, color="#ef4444", linewidth=2)
    
    # Spot price overlay
    ax1_price = ax1.twinx()
    valid_spots = [(t, s) for t, s in zip(times, spot_prices) if s]
    if valid_spots:
        st, sv = zip(*valid_spots)
        ax1_price.plot(st, sv, color="white", linewidth=1.5, alpha=0.7, label=f"Spot (${sv[-1]:.2f})")
        ax1_price.set_ylabel("Underlying ($)", fontsize=11, color="white")
        ax1_price.tick_params(axis="y", colors="white")
    
    ax1.set_ylabel(f"Cumulative Premium (${scale_label})", fontsize=12)
    
    # Determine bias
    bias = "BULLISH" if cum_call > cum_put else "BEARISH"
    ratio = cum_call / cum_put if cum_put > 0 else float('inf')
    ax1.set_title(f"Net Flow (Aggressive Buys Only) — {TRADE_DATE} | {TIME_LABEL}\n"
                  f"at_ask + above_ask trades only | Call:Put Ratio = {ratio:.2f}:1 ({bias})",
                  fontsize=14, color="white", pad=15, fontweight="bold")
    ax1.tick_params(colors="white")
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax1.grid(axis="both", alpha=0.2, color="#4a5568")
    
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_price.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", ncol=3, fontsize=9,
               framealpha=0.8, facecolor="#1a2744" if DARK_THEME else "white")
    
    # Panel 2: Per-minute volume bars
    bar_width = 0.0003
    ax2.bar([t - pd.Timedelta(seconds=15) for t in times], call_vol_minute, width=bar_width, 
            color="#22c55e", alpha=0.7, label="Call Vol")
    ax2.bar([t + pd.Timedelta(seconds=15) for t in times], put_vol_minute, width=bar_width, 
            color="#ef4444", alpha=0.7, label="Put Vol")
    ax2.set_ylabel("Volume/min", fontsize=10)
    ax2.set_xlabel("Time (ET)", fontsize=11)
    ax2.set_title("Per-Minute Aggressive Buy Volume", fontsize=11, color="white", pad=5)
    ax2.tick_params(colors="white", labelsize=9)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax2.grid(axis="both", alpha=0.2, color="#4a5568")
    ax2.legend(loc="upper right", fontsize=9, framealpha=0.8, facecolor="#1a2744")
    
    total_call_vol = sum(call_vol_minute)
    total_put_vol = sum(put_vol_minute)
    ax2.text(0.02, 0.95, f"Total: {total_call_vol:,} calls | {total_put_vol:,} puts", 
             transform=ax2.transAxes, fontsize=10, verticalalignment="top",
             bbox=dict(boxstyle="round", facecolor="#1a2744" if DARK_THEME else "white", alpha=0.8))
    
    for ax in [ax1, ax2]:
        for spine in ax.spines.values():
            spine.set_color("#4a5568")
    
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom", 
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout()
    plt.show()
    
    # Summary
    print(f"\nNet Flow Summary (Aggressive Buys Only):")
    print(f"  Call Flow: ${cum_call:,.0f} ({total_call_vol:,} contracts)")
    print(f"  Put Flow:  ${cum_put:,.0f} ({total_put_vol:,} contracts)")
    print(f"  Bias: {bias} (Call:Put = {ratio:.2f}:1)")
else:
    print("No aggressive buy data available")

In [ ]:
# ============================================================
# 4. COMBINED GEX + DEX CHART (Side-Weighted Only)
# ============================================================
# Single-column layout with:
# - Panel 1: Net GEX by strike with Spot, Abs, P1, N1 reference lines
# - Panel 2: Call vs Put GEX breakdown
# - Panel 3: Net DEX (Delta Exposure) by strike
from scipy.ndimage import gaussian_filter1d

STRIKE_RANGE_FROM_SPOT = 50
VOLUME_SMOOTHING_SIGMA = 1.5

# Use result.by_strike from calculate_gex (full range for key level calculations)
df_gex_full = result.by_strike.copy()
spot = result.spot

# Calculate representation scaling factors
# Base GEX formula: gamma * size * spot² * 0.01 (per 1% move)
# Base DEX formula: delta * size * spot * 100 (notional)
if GEX_REPRESENTATION == "pct":
    # Per 1% Move (current default) - no scaling needed
    gex_scale = 1.0
    dex_scale = 1.0
    y_unit = "$ Millions"
elif GEX_REPRESENTATION == "dollar":
    # Per $1 Move - divide by (spot * 0.01)
    gex_scale = 1.0 / (spot * 0.01)
    dex_scale = 1.0 / (spot * 0.01)
    y_unit = "$ Millions"
elif GEX_REPRESENTATION == "unit":
    # Per 1 Unit (raw) - divide by (spot² * 0.01) for GEX, (spot * 0.01) for DEX
    gex_scale = 1.0 / (spot * spot * 0.01)
    dex_scale = 1.0 / (spot * 0.01)
    y_unit = "Units"
else:
    gex_scale = 1.0
    dex_scale = 1.0
    y_unit = "$ Millions"

print(f"Representation: {REP_LABEL} (GEX scale: {gex_scale:.6f}, DEX scale: {dex_scale:.6f})")

# Calculate key GEX levels using FULL strike range (before filtering for display)
# Apply scaling for consistent key level calculations
df_gex_full['sw_gex_scaled'] = df_gex_full['sw_gex'] * gex_scale
df_gex_full['sw_call_gex_scaled'] = df_gex_full['sw_call_gex'] * gex_scale
df_gex_full['sw_put_gex_scaled'] = df_gex_full['sw_put_gex'] * gex_scale

# P1: Strike with highest positive GEX
positive_gex_full = df_gex_full[df_gex_full['sw_gex_scaled'] > 0]
p1_strike = positive_gex_full.loc[positive_gex_full['sw_gex_scaled'].idxmax(), 'strike'] if len(positive_gex_full) > 0 else None
p1_gex = positive_gex_full['sw_gex_scaled'].max() / 1e6 if len(positive_gex_full) > 0 else 0

# N1: Strike with most negative GEX
negative_gex_full = df_gex_full[df_gex_full['sw_gex_scaled'] < 0]
n1_strike = negative_gex_full.loc[negative_gex_full['sw_gex_scaled'].idxmin(), 'strike'] if len(negative_gex_full) > 0 else None
n1_gex = negative_gex_full['sw_gex_scaled'].min() / 1e6 if len(negative_gex_full) > 0 else 0

# Abs (GEX flip): where cumulative GEX crosses zero (using ALL strikes)
df_gex_sorted = df_gex_full.sort_values('strike')
cumulative_gex = df_gex_sorted['sw_gex_scaled'].cumsum()
sign_changes = cumulative_gex.shift(1) * cumulative_gex < 0
abs_strike = None
if sign_changes.any():
    flip_idx = sign_changes.idxmax()
    abs_strike = df_gex_sorted.loc[flip_idx, 'strike']

print(f"Key levels (from full strike range):")
print(f"  P1: ${p1_strike:,.0f} ({p1_gex:.1f}M)" if p1_strike else "  P1: None")
print(f"  N1: ${n1_strike:,.0f} ({n1_gex:.1f}M)" if n1_strike else "  N1: None")
print(f"  Abs: ${abs_strike:,.0f}" if abs_strike else "  Abs: None (no zero crossing)")

# Now filter for display
if STRIKE_RANGE_FROM_SPOT and spot:
    df_gex = df_gex_full[
        (df_gex_full['strike'] >= spot - STRIKE_RANGE_FROM_SPOT) &
        (df_gex_full['strike'] <= spot + STRIKE_RANGE_FROM_SPOT)
    ].copy()
    range_label = f"±${STRIKE_RANGE_FROM_SPOT} from spot"
else:
    df_gex = df_gex_full.copy()
    range_label = "All Strikes"

print(f"GEX chart: {len(df_gex)} strikes within {range_label}")

if len(df_gex) > 0:
    strikes = df_gex['strike'].values
    bar_width = (strikes[-1] - strikes[0]) / len(strikes) * 0.8 if len(strikes) > 1 else 5

    # Side-weighted GEX (scaled)
    sw_net_gex_m = df_gex['sw_gex_scaled'].values / 1_000_000
    sw_call_gex_m = df_gex['sw_call_gex_scaled'].values / 1_000_000
    sw_put_gex_m = df_gex['sw_put_gex_scaled'].values / 1_000_000
    total_sw_gex = df_gex['sw_gex_scaled'].sum()

    # Calculate DEX (Delta Exposure) by strike
    # First, calculate Greeks for df if not already present
    df_dex = df.copy()
    if 'delta' not in df_dex.columns:
        greeks = calculate_greeks(
            spot=df_dex['spot'].values,
            strike=df_dex['strike'].values,
            tte=df_dex['tte_years'].values,
            rate=0.05,
            price=df_dex['price'].values,
            is_call=(df_dex['opt_type'] == 'C').values
        )
        df_dex['delta'] = greeks['delta']
    
    # DEX = delta * size * spot * 100 (side-weighted), then apply scale
    # Side-weighted: flip sign for SELL trades (MM perspective)
    df_dex['dex'] = df_dex['delta'] * df_dex['size'] * df_dex['spot'] * 100 * dex_scale
    df_dex.loc[df_dex['trade_dir'] == 'SELL', 'dex'] *= -1

    # Filter to strike range
    df_dex = df_dex[(df_dex['strike'] >= spot - STRIKE_RANGE_FROM_SPOT) &
                    (df_dex['strike'] <= spot + STRIKE_RANGE_FROM_SPOT)]

    # Aggregate NET DEX by strike (sum of call + put DEX per strike)
    net_dex_by_strike = df_dex.groupby('strike').agg({'dex': 'sum'}).reset_index()
    net_dex_by_strike = net_dex_by_strike.set_index('strike').reindex(df_gex['strike']).fillna(0)
    net_dex_m = net_dex_by_strike['dex'].values / 1_000_000
    total_dex = net_dex_m.sum()

    # Aggregate volume by strike
    vol_by_strike = df.groupby(['strike', 'trade_dir']).agg({'size': 'sum'}).reset_index()
    vol_pivot = vol_by_strike.pivot(index='strike', columns='trade_dir', values='size').fillna(0)
    vol_pivot = vol_pivot.reindex(df_gex['strike']).fillna(0)

    bullish_vol = vol_pivot.get('BUY', pd.Series(0, index=df_gex['strike'])).values.astype(float)
    bearish_vol = vol_pivot.get('SELL', pd.Series(0, index=df_gex['strike'])).values.astype(float)
    total_vol = bullish_vol + bearish_vol

    if VOLUME_SMOOTHING_SIGMA > 0 and len(total_vol) > 3:
        total_vol_smooth = gaussian_filter1d(total_vol, sigma=VOLUME_SMOOTHING_SIGMA)
        bullish_vol_smooth = gaussian_filter1d(bullish_vol, sigma=VOLUME_SMOOTHING_SIGMA)
        bearish_vol_smooth = gaussian_filter1d(bearish_vol, sigma=VOLUME_SMOOTHING_SIGMA)
    else:
        total_vol_smooth = total_vol
        bullish_vol_smooth = bullish_vol
        bearish_vol_smooth = bearish_vol

    # Create figure with 3 rows (single column)
    if DARK_THEME:
        plt.style.use("dark_background")

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 18), height_ratios=[2, 1, 1],
                                         gridspec_kw={"hspace": 0.18})

    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2, ax3]:
            ax.set_facecolor("#0a1628")

    # PANEL 1: Net GEX with Reference Lines
    ax1_vol = ax1.twinx()

    # Volume fill
    ax1_vol.fill_between(strikes, bullish_vol_smooth, alpha=0.15, color='#22c55e', zorder=1)
    ax1_vol.fill_between(strikes, bearish_vol_smooth, alpha=0.15, color='#ef4444', zorder=1)
    ax1_vol.plot(strikes, bullish_vol_smooth, color='#22c55e', linewidth=1.2, alpha=0.5, zorder=1)
    ax1_vol.plot(strikes, bearish_vol_smooth, color='#ef4444', linewidth=1.2, alpha=0.5, zorder=1)

    ax1.axhline(y=0, color='#4a5568', linestyle='-', linewidth=0.5, alpha=0.5, zorder=2)
    bar_colors = ['#22c55e' if g >= 0 else '#ef4444' for g in sw_net_gex_m]
    ax1.bar(strikes, sw_net_gex_m, width=bar_width, color=bar_colors, alpha=0.85, zorder=3,
            label=f'Net GEX ({total_sw_gex/1e6:.1f}M)')

    # Reference lines with labels
    y_lim = ax1.get_ylim()
    y_max = max(abs(y_lim[0]), abs(y_lim[1]))
    ax1.set_ylim(-y_max * 1.1, y_max * 1.1)
    y_max = ax1.get_ylim()[1]

    # Spot line (red solid)
    if spot:
        ax1.axvline(x=spot, color='#ef4444', linestyle='-', linewidth=2.5, zorder=10)
        ax1.text(spot, y_max * 0.95, f'Spot\n${spot:,.0f}', color='white', fontsize=9, fontweight='bold',
                 ha='center', va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='#ef4444', alpha=0.9))

    # Abs line (purple dashed) - GEX flip point (only show if within display range)
    if abs_strike and strikes.min() <= abs_strike <= strikes.max():
        ax1.axvline(x=abs_strike, color='#a855f7', linestyle='--', linewidth=2, zorder=9)
        ax1.text(abs_strike, y_max * 0.75, f'Abs\n${abs_strike:,.0f}', color='white', fontsize=9, fontweight='bold',
                 ha='center', va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='#a855f7', alpha=0.9))

    # P1 line (green dashed) - highest positive GEX (only show if within display range)
    if p1_strike and strikes.min() <= p1_strike <= strikes.max():
        ax1.axvline(x=p1_strike, color='#22c55e', linestyle='--', linewidth=2, zorder=9)
        ax1.text(p1_strike, y_max * 0.55, f'P1\n${p1_strike:,.0f}', color='white', fontsize=9, fontweight='bold',
                 ha='center', va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='#22c55e', alpha=0.9))

    # N1 line (orange dashed) - most negative GEX (only show if within display range)
    if n1_strike and strikes.min() <= n1_strike <= strikes.max():
        ax1.axvline(x=n1_strike, color='#f97316', linestyle='--', linewidth=2, zorder=9)
        ax1.text(n1_strike, y_max * 0.35, f'N1\n${n1_strike:,.0f}', color='white', fontsize=9, fontweight='bold',
                 ha='center', va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='#f97316', alpha=0.9))

    ax1.set_ylabel(f'Net GEX ({y_unit})', fontsize=12, color='white')
    ax1_vol.set_ylabel('Volume', fontsize=11, color='#3b82f6')
    ax1_vol.tick_params(axis='y', colors='#3b82f6')

    sw_effect = "STABILIZING" if total_sw_gex > 0 else "DESTABILIZING"
    ax1.set_title(f'Side-Weighted GEX — Net: {total_sw_gex/1e6:.2f}M ({sw_effect})\n'
                  f'Buy Rate: Calls {result.call_buy_pct:.0f}% | Puts {result.put_buy_pct:.0f}%',
                  fontsize=14, fontweight='bold', color='white', pad=10)

    ax1.tick_params(colors='white')
    ax1.grid(axis='y', alpha=0.2, color='#4a5568')
    for spine in ax1.spines.values(): spine.set_color('#4a5568')

    # PANEL 2: Call vs Put GEX Breakdown
    ax2.bar(strikes, sw_call_gex_m, width=bar_width, color='#22c55e', alpha=0.8,
            label=f'Call GEX ({result.sw_call * gex_scale/1e6:.1f}M)')
    ax2.bar(strikes, sw_put_gex_m, width=bar_width, color='#ef4444', alpha=0.8,
            label=f'Put GEX ({result.sw_put * gex_scale/1e6:.1f}M)')
    ax2.axhline(y=0, color='#4a5568', linestyle='-', linewidth=0.5, alpha=0.5)

    # Reference lines (no labels)
    if spot:
        ax2.axvline(x=spot, color='#ef4444', linestyle='-', linewidth=2, alpha=0.7)
    if abs_strike and strikes.min() <= abs_strike <= strikes.max():
        ax2.axvline(x=abs_strike, color='#a855f7', linestyle='--', linewidth=1.5, alpha=0.7)
    if p1_strike and strikes.min() <= p1_strike <= strikes.max():
        ax2.axvline(x=p1_strike, color='#22c55e', linestyle='--', linewidth=1.5, alpha=0.7)
    if n1_strike and strikes.min() <= n1_strike <= strikes.max():
        ax2.axvline(x=n1_strike, color='#f97316', linestyle='--', linewidth=1.5, alpha=0.7)

    ax2.set_ylabel(f'GEX ({y_unit})', fontsize=11, color='white')
    ax2.set_title('GEX Breakdown: Calls (+) vs Puts (-)', fontsize=12, color='white', pad=5)
    ax2.legend(loc='upper right', fontsize=10, framealpha=0.8, facecolor='#1a2744')
    ax2.tick_params(colors='white')
    ax2.grid(axis='y', alpha=0.2, color='#4a5568')
    for spine in ax2.spines.values(): spine.set_color('#4a5568')

    # PANEL 3: Net DEX (Delta Exposure) by strike
    dex_colors = ['#22c55e' if d >= 0 else '#ef4444' for d in net_dex_m]
    ax3.bar(strikes, net_dex_m, width=bar_width, color=dex_colors, alpha=0.85,
            label=f'Net DEX ({total_dex:.1f}M)')
    ax3.axhline(y=0, color='#4a5568', linestyle='-', linewidth=0.5, alpha=0.5)

    # Reference lines (no labels)
    if spot:
        ax3.axvline(x=spot, color='#ef4444', linestyle='-', linewidth=2, alpha=0.7)
    if abs_strike and strikes.min() <= abs_strike <= strikes.max():
        ax3.axvline(x=abs_strike, color='#a855f7', linestyle='--', linewidth=1.5, alpha=0.7)
    if p1_strike and strikes.min() <= p1_strike <= strikes.max():
        ax3.axvline(x=p1_strike, color='#22c55e', linestyle='--', linewidth=1.5, alpha=0.7)
    if n1_strike and strikes.min() <= n1_strike <= strikes.max():
        ax3.axvline(x=n1_strike, color='#f97316', linestyle='--', linewidth=1.5, alpha=0.7)

    ax3.set_ylabel(f'Net DEX ({y_unit})', fontsize=11, color='white')
    ax3.set_xlabel('Strike', fontsize=12, color='white')
    dex_bias = "LONG DELTA" if total_dex > 0 else "SHORT DELTA"
    ax3.set_title(f'Net DEX (Delta Exposure) — {total_dex:.1f}M ({dex_bias})', fontsize=12, color='white', pad=5)
    ax3.tick_params(colors='white')
    ax3.grid(axis='y', alpha=0.2, color='#4a5568')
    for spine in ax3.spines.values(): spine.set_color('#4a5568')

    # Set common x-axis ticks
    num_ticks = min(25, len(strikes))
    tick_indices = np.linspace(0, len(strikes)-1, num_ticks, dtype=int)
    for ax in [ax1, ax2, ax3]:
        ax.set_xticks(strikes[tick_indices])
        ax.set_xticklabels([f'{s:.0f}' for s in strikes[tick_indices]], rotation=45, ha='right', fontsize=8)

    fig.suptitle(f"Combined GEX + DEX Analysis — {TRADE_DATE} | {TIME_LABEL}\n{REP_LABEL} | {MONEYNESS_LABEL} | {range_label}",
                 fontsize=16, color="white", fontweight="bold", y=0.995)
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

    # Print key levels summary
    print(f"\nKey GEX Levels ({REP_LABEL}, full range):")
    print(f"  Spot:  ${spot:,.0f}")
    if abs_strike:
        print(f"  Abs:   ${abs_strike:,.0f} (GEX flip point)")
    if p1_strike:
        print(f"  P1:    ${p1_strike:,.0f} ({p1_gex:.1f}M - highest positive GEX)")
    if n1_strike:
        print(f"  N1:    ${n1_strike:,.0f} ({n1_gex:.1f}M - most negative GEX)")
    print(f"\nNet DEX: {total_dex:.1f}M ({dex_bias})")
else:
    print("No GEX data available")

In [ ]:
# ============================================================
# 4b. VOMMA/ZOMMA EXPOSURE BY STRIKE (Higher-Order Greeks)
# ============================================================
# Vomma = d(vega)/d(sigma) - how vega changes with volatility
# Zomma = d(gamma)/d(sigma) - how gamma changes with volatility
#
# DIRECTIONAL INTERPRETATION (after side-weighting):
#   Positive zomma_exp = DESTABILIZING (vol spike increases dealer hedging pressure)
#   Negative zomma_exp = STABILIZING (vol spike decreases dealer hedging pressure)
#
# Sign convention: BUY trades keep zomma sign, SELL trades flip it
# This reflects dealer positioning (opposite of customer)

from scipy.stats import norm

# Extended Greeks functions (vectorized)
def d1_func(spot, strike, tte, rate, sigma):
    tte_safe = np.maximum(tte, 1e-10)
    sigma_safe = np.maximum(sigma, 1e-10)
    return (np.log(spot / strike) + (rate + 0.5 * sigma_safe**2) * tte_safe) / (sigma_safe * np.sqrt(tte_safe))

def d2_func(d1, sigma, tte):
    return d1 - sigma * np.sqrt(np.maximum(tte, 1e-10))

def calc_vega(spot, strike, tte, rate, sigma):
    """Vega = S * sqrt(T) * N'(d1)"""
    d1 = d1_func(spot, strike, tte, rate, sigma)
    return spot * np.sqrt(np.maximum(tte, 1e-10)) * norm.pdf(d1)

def calc_vomma(spot, strike, tte, rate, sigma):
    """Vomma = vega * d1 * d2 / sigma"""
    sig = np.maximum(sigma, 1e-10)
    d1 = d1_func(spot, strike, tte, rate, sig)
    d2 = d2_func(d1, sig, tte)
    v = calc_vega(spot, strike, tte, rate, sig)
    return v * d1 * d2 / sig

def calc_gamma(spot, strike, tte, rate, sigma):
    """Gamma = N'(d1) / (S * sigma * sqrt(T))"""
    tte_safe = np.maximum(tte, 1e-10)
    sig = np.maximum(sigma, 1e-10)
    d1 = d1_func(spot, strike, tte_safe, rate, sig)
    return norm.pdf(d1) / (spot * sig * np.sqrt(tte_safe))

def calc_zomma(spot, strike, tte, rate, sigma):
    """Zomma = gamma * (d1*d2 - 1) / sigma"""
    sig = np.maximum(sigma, 1e-10)
    d1 = d1_func(spot, strike, tte, rate, sig)
    d2 = d2_func(d1, sig, tte)
    g = calc_gamma(spot, strike, tte, rate, sig)
    return g * (d1 * d2 - 1) / sig

# Calculate Greeks for all trades
df_greeks = df.copy()
greeks_result = calculate_greeks(
    spot=df_greeks['spot'].values,
    strike=df_greeks['strike'].values,
    tte=df_greeks['tte_years'].values,
    rate=0.05,
    price=df_greeks['price'].values,
    is_call=(df_greeks['opt_type'] == 'C').values
)
df_greeks['iv'] = greeks_result['iv']

# Filter valid IV
df_greeks = df_greeks[(df_greeks['iv'] > 0.05) & (df_greeks['iv'] < 2.0)]
print(f"Valid trades for vomma/zomma: {len(df_greeks):,} ({len(df_greeks)/len(df)*100:.1f}%)")

# Calculate vomma and zomma
df_greeks['vomma'] = calc_vomma(
    df_greeks['spot'].values, df_greeks['strike'].values,
    df_greeks['tte_years'].values, 0.05, df_greeks['iv'].values
)
df_greeks['zomma'] = calc_zomma(
    df_greeks['spot'].values, df_greeks['strike'].values,
    df_greeks['tte_years'].values, 0.05, df_greeks['iv'].values
)

# Calculate exposure: greek * size * 100 (notional multiplier)
# Side-weighted: flip sign for SELL trades (MM perspective)
df_greeks['vomma_exp'] = df_greeks['vomma'] * df_greeks['size'] * 100
df_greeks['zomma_exp'] = df_greeks['zomma'] * df_greeks['size'] * 100
df_greeks.loc[df_greeks['trade_dir'] == 'SELL', 'vomma_exp'] *= -1
df_greeks.loc[df_greeks['trade_dir'] == 'SELL', 'zomma_exp'] *= -1

# Filter to strike range for display
spot = result.spot
df_greeks_display = df_greeks[
    (df_greeks['strike'] >= spot - STRIKE_RANGE_FROM_SPOT) &
    (df_greeks['strike'] <= spot + STRIKE_RANGE_FROM_SPOT)
].copy()

# Aggregate by strike
vomma_by_strike = df_greeks_display.groupby(['strike', 'opt_type']).agg({
    'vomma_exp': 'sum',
    'zomma_exp': 'sum',
    'size': 'sum'
}).reset_index()

# Pivot for call/put breakdown
vomma_call = vomma_by_strike[vomma_by_strike['opt_type'] == 'C'].set_index('strike')['vomma_exp']
vomma_put = vomma_by_strike[vomma_by_strike['opt_type'] == 'P'].set_index('strike')['vomma_exp']
zomma_call = vomma_by_strike[vomma_by_strike['opt_type'] == 'C'].set_index('strike')['zomma_exp']
zomma_put = vomma_by_strike[vomma_by_strike['opt_type'] == 'P'].set_index('strike')['zomma_exp']

# Net by strike
net_vomma = df_greeks_display.groupby('strike')['vomma_exp'].sum()
net_zomma = df_greeks_display.groupby('strike')['zomma_exp'].sum()

# Create figure with 3 panels
if len(net_vomma) > 0:
    strikes = net_vomma.index.values
    bar_width = (strikes[-1] - strikes[0]) / len(strikes) * 0.8 if len(strikes) > 1 else 5
    
    if DARK_THEME:
        plt.style.use("dark_background")
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 16), height_ratios=[2, 1, 1],
                                         gridspec_kw={"hspace": 0.18})
    
    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2, ax3]:
            ax.set_facecolor("#0a1628")
    
    # Scale for display (thousands)
    scale = 1000
    net_vomma_k = net_vomma.values / scale
    net_zomma_k = net_zomma.values / scale
    total_vomma = net_vomma.sum() / scale
    total_zomma = net_zomma.sum() / scale
    
    # PANEL 1: Net Vomma with volume overlay
    ax1_vol = ax1.twinx()
    
    # Volume by strike
    vol_by_strike = df_greeks_display.groupby('strike')['size'].sum()
    vol_smooth = gaussian_filter1d(vol_by_strike.reindex(strikes).fillna(0).values.astype(float), sigma=1.5)
    ax1_vol.fill_between(strikes, vol_smooth, alpha=0.15, color='#3b82f6', zorder=1)
    ax1_vol.plot(strikes, vol_smooth, color='#3b82f6', linewidth=1.2, alpha=0.5, zorder=1)
    
    ax1.axhline(y=0, color='#4a5568', linestyle='-', linewidth=0.5, alpha=0.5, zorder=2)
    vomma_colors = ['#22c55e' if v >= 0 else '#ef4444' for v in net_vomma_k]
    ax1.bar(strikes, net_vomma_k, width=bar_width, color=vomma_colors, alpha=0.85, zorder=3)
    
    # Reference lines
    y_lim = ax1.get_ylim()
    y_max = max(abs(y_lim[0]), abs(y_lim[1]))
    ax1.set_ylim(-y_max * 1.1, y_max * 1.1)
    y_max = ax1.get_ylim()[1]
    
    # Spot line
    ax1.axvline(x=spot, color='#ef4444', linestyle='-', linewidth=2.5, zorder=10)
    ax1.text(spot, y_max * 0.95, f'Spot\n${spot:,.0f}', color='white', fontsize=9, fontweight='bold',
             ha='center', va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='#ef4444', alpha=0.9))
    
    # P1/N1 from GEX (if available)
    if p1_strike and strikes.min() <= p1_strike <= strikes.max():
        ax1.axvline(x=p1_strike, color='#22c55e', linestyle='--', linewidth=1.5, alpha=0.5, zorder=8)
    if n1_strike and strikes.min() <= n1_strike <= strikes.max():
        ax1.axvline(x=n1_strike, color='#f97316', linestyle='--', linewidth=1.5, alpha=0.5, zorder=8)
    
    ax1.set_ylabel('Net Vomma (K)', fontsize=12, color='white')
    ax1_vol.set_ylabel('Volume', fontsize=11, color='#3b82f6')
    ax1_vol.tick_params(axis='y', colors='#3b82f6')
    
    vomma_bias = "VOL ACCELERATION" if total_vomma > 0 else "VOL DAMPENING"
    ax1.set_title(f'Side-Weighted Vomma Exposure — Net: {total_vomma:,.0f}K ({vomma_bias})\n'
                  f'Positive = Vol increases amplify vega | Negative = Vol increases dampen vega',
                  fontsize=14, fontweight='bold', color='white', pad=10)
    ax1.tick_params(colors='white')
    ax1.grid(axis='y', alpha=0.2, color='#4a5568')
    for spine in ax1.spines.values(): spine.set_color('#4a5568')
    
    # PANEL 2: Call vs Put Vomma Breakdown
    call_vomma_k = vomma_call.reindex(strikes).fillna(0).values / scale
    put_vomma_k = vomma_put.reindex(strikes).fillna(0).values / scale
    
    ax2.bar(strikes, call_vomma_k, width=bar_width, color='#22c55e', alpha=0.8,
            label=f'Call Vomma ({vomma_call.sum()/scale:,.0f}K)')
    ax2.bar(strikes, put_vomma_k, width=bar_width, color='#ef4444', alpha=0.8,
            label=f'Put Vomma ({vomma_put.sum()/scale:,.0f}K)')
    ax2.axhline(y=0, color='#4a5568', linestyle='-', linewidth=0.5, alpha=0.5)
    
    ax2.axvline(x=spot, color='#ef4444', linestyle='-', linewidth=2, alpha=0.7)
    ax2.set_ylabel('Vomma (K)', fontsize=11, color='white')
    ax2.set_title('Vomma Breakdown: Calls vs Puts', fontsize=12, color='white', pad=5)
    ax2.legend(loc='upper right', fontsize=10, framealpha=0.8, facecolor='#1a2744')
    ax2.tick_params(colors='white')
    ax2.grid(axis='y', alpha=0.2, color='#4a5568')
    for spine in ax2.spines.values(): spine.set_color('#4a5568')
    
    # PANEL 3: Net Zomma
    zomma_colors = ['#22c55e' if z >= 0 else '#ef4444' for z in net_zomma_k]
    ax3.bar(strikes, net_zomma_k, width=bar_width, color=zomma_colors, alpha=0.85)
    ax3.axhline(y=0, color='#4a5568', linestyle='-', linewidth=0.5, alpha=0.5)
    ax3.axvline(x=spot, color='#ef4444', linestyle='-', linewidth=2, alpha=0.7)
    
    ax3.set_ylabel('Net Zomma (K)', fontsize=11, color='white')
    ax3.set_xlabel('Strike', fontsize=12, color='white')
    zomma_bias = "GAMMA INFLATION" if total_zomma > 0 else "GAMMA DEFLATION"
    ax3.set_title(f'Net Zomma Exposure — {total_zomma:,.0f}K ({zomma_bias})\n'
                  f'Positive = Vol increase inflates gamma | Negative = Vol increase deflates gamma',
                  fontsize=12, color='white', pad=5)
    ax3.tick_params(colors='white')
    ax3.grid(axis='y', alpha=0.2, color='#4a5568')
    for spine in ax3.spines.values(): spine.set_color('#4a5568')
    
    # Common x-axis ticks
    num_ticks = min(25, len(strikes))
    tick_indices = np.linspace(0, len(strikes)-1, num_ticks, dtype=int)
    for ax in [ax1, ax2, ax3]:
        ax.set_xticks(strikes[tick_indices])
        ax.set_xticklabels([f'{s:.0f}' for s in strikes[tick_indices]], rotation=45, ha='right', fontsize=8)
    
    fig.suptitle(f"Vomma/Zomma Exposure Analysis — {TRADE_DATE} | {TIME_LABEL}\n"
                 f"{MONEYNESS_LABEL} | ±${STRIKE_RANGE_FROM_SPOT} from spot",
                 fontsize=16, color="white", fontweight="bold", y=0.995)
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()
    
    # Summary
    print(f"\nVomma/Zomma Summary:")
    print(f"  Net Vomma: {total_vomma:,.0f}K ({vomma_bias})")
    print(f"    Calls: {vomma_call.sum()/scale:,.0f}K | Puts: {vomma_put.sum()/scale:,.0f}K")
    print(f"  Net Zomma: {total_zomma:,.0f}K ({zomma_bias})")
    print(f"    Calls: {zomma_call.sum()/scale:,.0f}K | Puts: {zomma_put.sum()/scale:,.0f}K")
else:
    print("Insufficient data for vomma/zomma chart")

In [ ]:
# ============================================================
# 4c. VOMMA/ZOMMA INTERVAL EXPOSURE MAP (Bubble Chart)
# ============================================================
# Shows Vomma and Zomma evolution over time by strike
# - X-axis: Time intervals (configurable, default 5-min)
# - Y-axis: Strike price
# - Bubble size: Greek magnitude
# - Bubble color: RED (positive = DESTABILIZING), GREEN (negative = STABILIZING)
# - Yellow line: Underlying price evolution
#
# DIRECTIONAL INTERPRETATION:
#   RED bubbles = Vol spike increases dealer hedging pressure (destabilizing)
#   GREEN bubbles = Vol spike decreases dealer hedging pressure (stabilizing)

VOMMA_INTERVAL_MINUTES = 5
VOMMA_INTERVAL_STRIKE_RANGE = 50  # ±$50 from spot

# Use df_greeks from the previous Vomma/Zomma cell (already has vomma_exp, zomma_exp calculated)
df_vomma_interval = df_greeks.copy()

# Create time intervals
df_vomma_interval['interval'] = df_vomma_interval['timestamp'].dt.floor(f'{VOMMA_INTERVAL_MINUTES}min')

# Filter to strike range
spot = result.spot
df_vomma_interval = df_vomma_interval[
    (df_vomma_interval['strike'] >= spot - VOMMA_INTERVAL_STRIKE_RANGE) &
    (df_vomma_interval['strike'] <= spot + VOMMA_INTERVAL_STRIKE_RANGE)
].copy()

# Aggregate by interval and strike
interval_vomma = df_vomma_interval.groupby(['interval', 'strike']).agg({
    'vomma_exp': 'sum',
    'zomma_exp': 'sum',
    'spot': 'last',
    'size': 'sum'
}).reset_index()
interval_vomma.columns = ['interval_dt', 'strike', 'net_vomma', 'net_zomma', 'avg_spot', 'volume']

# Convert to ET for display
interval_vomma['interval_dt'] = interval_vomma['interval_dt'].dt.tz_convert(ET)

print(f"Vomma/Zomma Interval data: {len(interval_vomma):,} interval-strike combinations")
print(f"Time range: {interval_vomma['interval_dt'].min()} to {interval_vomma['interval_dt'].max()}")

if len(interval_vomma) > 0:
    # Create figure with 2 panels (Vomma and Zomma)
    if DARK_THEME:
        plt.style.use("dark_background")
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 20), height_ratios=[1, 1],
                                    gridspec_kw={"hspace": 0.15})
    
    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2]:
            ax.set_facecolor("#0a1628")
    
    # ========== PANEL 1: VOMMA INTERVAL MAP ==========
    max_vomma = interval_vomma['net_vomma'].abs().max()
    min_size = 10
    max_size = 300
    max_abs_vomma = max(abs(interval_vomma['net_vomma'].min()), abs(interval_vomma['net_vomma'].max()))
    
    # Plot Vomma bubbles
    for _, row in interval_vomma.iterrows():
        x = row['interval_dt']
        y = row['strike']
        vomma = row['net_vomma']
        
        size = min_size + (abs(vomma) / max_vomma) * (max_size - min_size) if max_vomma > 0 else min_size
        
        if vomma > 0:
            color = '#22c55e'  # Green - vol acceleration
            alpha = min(0.9, 0.3 + 0.6 * (vomma / max_abs_vomma)) if max_abs_vomma > 0 else 0.5
        else:
            color = '#ef4444'  # Red - vol dampening
            alpha = min(0.9, 0.3 + 0.6 * (abs(vomma) / max_abs_vomma)) if max_abs_vomma > 0 else 0.5
        
        ax1.scatter(x, y, s=size, c=color, alpha=alpha, edgecolors='white', linewidths=0.3)
    
    # Underlying price line
    spot_by_interval = interval_vomma.groupby('interval_dt')['avg_spot'].mean().dropna()
    if len(spot_by_interval) > 0:
        ax1.plot(spot_by_interval.index, spot_by_interval.values,
                 color='#fbbf24', linewidth=2.5, alpha=0.9, label='Underlying Price', zorder=10)
    
    # Current spot line
    ax1.axhline(y=spot, color='#ef4444', linewidth=2, linestyle='--', alpha=0.8,
                label=f'Current Spot: ${spot:,.0f}', zorder=9)
    
    ax1.set_ylabel('Strike Price', fontsize=12, color='white')
    total_vomma = interval_vomma['net_vomma'].sum()
    vomma_bias = "VOL ACCELERATION" if total_vomma > 0 else "VOL DAMPENING"
    ax1.set_title(f"Vomma Interval Map — Net: {total_vomma/1000:,.0f}K ({vomma_bias})\n"
                  f"Green = Vol increases amplify vega | Red = Vol increases dampen vega",
                  fontsize=14, fontweight='bold', color='white', pad=10)
    
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax1.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30], tz=ET))
    
    # Y-axis ticks
    all_strikes = sorted(interval_vomma['strike'].unique())
    if len(all_strikes) > 0:
        strike_range = [min(all_strikes), max(all_strikes)]
        y_ticks = np.arange(int(strike_range[0] // 5) * 5, int(strike_range[1] // 5 + 1) * 5, 5)
        ax1.set_yticks(y_ticks)
    
    ax1.grid(axis='both', alpha=0.2, color='#4a5568')
    ax1.tick_params(colors='white')
    for spine in ax1.spines.values():
        spine.set_color('#4a5568')
    
    # Vomma legend
    vomma_legend = [
        plt.scatter([], [], s=30, c='#22c55e', alpha=0.7, edgecolors='white', label='Small + Vomma'),
        plt.scatter([], [], s=150, c='#22c55e', alpha=0.7, edgecolors='white', label='Large + Vomma'),
        plt.scatter([], [], s=30, c='#ef4444', alpha=0.7, edgecolors='white', label='Small - Vomma'),
        plt.scatter([], [], s=150, c='#ef4444', alpha=0.7, edgecolors='white', label='Large - Vomma'),
        plt.Line2D([0], [0], color='#fbbf24', linewidth=2.5, label='Underlying Price'),
    ]
    ax1.legend(handles=vomma_legend, loc='upper left', fontsize=9, framealpha=0.8,
               facecolor='#1a2744' if DARK_THEME else 'white', title='Vomma Magnitude')
    
    # ========== PANEL 2: ZOMMA INTERVAL MAP ==========
    max_zomma = interval_vomma['net_zomma'].abs().max()
    max_abs_zomma = max(abs(interval_vomma['net_zomma'].min()), abs(interval_vomma['net_zomma'].max()))
    
    # Plot Zomma bubbles
    for _, row in interval_vomma.iterrows():
        x = row['interval_dt']
        y = row['strike']
        zomma = row['net_zomma']
        
        size = min_size + (abs(zomma) / max_zomma) * (max_size - min_size) if max_zomma > 0 else min_size
        
        if zomma > 0:
            color = '#22c55e'  # Green - gamma inflation
            alpha = min(0.9, 0.3 + 0.6 * (zomma / max_abs_zomma)) if max_abs_zomma > 0 else 0.5
        else:
            color = '#ef4444'  # Red - gamma deflation
            alpha = min(0.9, 0.3 + 0.6 * (abs(zomma) / max_abs_zomma)) if max_abs_zomma > 0 else 0.5
        
        ax2.scatter(x, y, s=size, c=color, alpha=alpha, edgecolors='white', linewidths=0.3)
    
    # Underlying price line
    if len(spot_by_interval) > 0:
        ax2.plot(spot_by_interval.index, spot_by_interval.values,
                 color='#fbbf24', linewidth=2.5, alpha=0.9, label='Underlying Price', zorder=10)
    
    # Current spot line
    ax2.axhline(y=spot, color='#ef4444', linewidth=2, linestyle='--', alpha=0.8,
                label=f'Current Spot: ${spot:,.0f}', zorder=9)
    
    ax2.set_xlabel('Time (ET)', fontsize=12, color='white')
    ax2.set_ylabel('Strike Price', fontsize=12, color='white')
    total_zomma = interval_vomma['net_zomma'].sum()
    zomma_bias = "GAMMA INFLATION" if total_zomma > 0 else "GAMMA DEFLATION"
    ax2.set_title(f"Zomma Interval Map — Net: {total_zomma/1000:,.0f}K ({zomma_bias})\n"
                  f"Green = Vol increase inflates gamma | Red = Vol increase deflates gamma",
                  fontsize=14, fontweight='bold', color='white', pad=10)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax2.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30], tz=ET))
    plt.xticks(rotation=45)
    
    if len(all_strikes) > 0:
        ax2.set_yticks(y_ticks)
    
    ax2.grid(axis='both', alpha=0.2, color='#4a5568')
    ax2.tick_params(colors='white')
    for spine in ax2.spines.values():
        spine.set_color('#4a5568')
    
    # Zomma legend
    zomma_legend = [
        plt.scatter([], [], s=30, c='#22c55e', alpha=0.7, edgecolors='white', label='Small + Zomma'),
        plt.scatter([], [], s=150, c='#22c55e', alpha=0.7, edgecolors='white', label='Large + Zomma'),
        plt.scatter([], [], s=30, c='#ef4444', alpha=0.7, edgecolors='white', label='Small - Zomma'),
        plt.scatter([], [], s=150, c='#ef4444', alpha=0.7, edgecolors='white', label='Large - Zomma'),
        plt.Line2D([0], [0], color='#fbbf24', linewidth=2.5, label='Underlying Price'),
    ]
    ax2.legend(handles=zomma_legend, loc='upper left', fontsize=9, framealpha=0.8,
               facecolor='#1a2744' if DARK_THEME else 'white', title='Zomma Magnitude')
    
    fig.suptitle(f"Vomma/Zomma Interval Exposure Map — {TRADE_DATE} | {TIME_LABEL}\n"
                 f"{VOMMA_INTERVAL_MINUTES}-min intervals | ±${VOMMA_INTERVAL_STRIKE_RANGE} from spot | {MONEYNESS_LABEL}",
                 fontsize=16, color="white", fontweight="bold", y=0.995)
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()
    
    # Summary
    print(f"\nVomma/Zomma Interval Summary:")
    print(f"  Net Vomma: {total_vomma/1000:,.0f}K ({vomma_bias})")
    print(f"  Net Zomma: {total_zomma/1000:,.0f}K ({zomma_bias})")
    print(f"  Intervals: {interval_vomma['interval_dt'].nunique()}")
    print(f"  Strikes: {interval_vomma['strike'].nunique()}")
    
    # Find peak exposure times
    vomma_by_time = interval_vomma.groupby('interval_dt')['net_vomma'].sum()
    zomma_by_time = interval_vomma.groupby('interval_dt')['net_zomma'].sum()
    
    peak_vomma_time = vomma_by_time.abs().idxmax()
    peak_zomma_time = zomma_by_time.abs().idxmax()
    
    print(f"\n  Peak Vomma: {vomma_by_time[peak_vomma_time]/1000:,.0f}K at {peak_vomma_time.strftime('%H:%M')} ET")
    print(f"  Peak Zomma: {zomma_by_time[peak_zomma_time]/1000:,.0f}K at {peak_zomma_time.strftime('%H:%M')} ET")
else:
    print("No interval Vomma/Zomma data available")

In [ ]:
# ============================================================
# 4d. CONVEXITY ACCELERATION RISK (CAR) - Original Formula
# ============================================================
# ⚠️ NOTE: This uses the ORIGINAL formula with clip(lower=0).
#    See Cell 4e below for IMPROVED DIRECTIONAL ANALYSIS that properly
#    accounts for trade side (destabilizing vs stabilizing flow).
#
# Original Formula: CAR = -Γ_net × (0.6·Zomma+ + 0.4·Vomma+) × τ^(-0.5)
#
# KNOWN ISSUES with this approach:
#   1. clip(lower=0) loses signal when net zomma is negative
#   2. Vomma is largely irrelevant for 0DTE (vega→0 as τ→0)
#   3. Doesn't distinguish destabilizing from stabilizing flow
#
# Panel 1: CAR magnitude over time (aggregate)
# Panel 2: Strike-level CAR contribution bubble chart

CAR_INTERVAL_MINUTES = 5
CAR_STRIKE_RANGE = 50  # ±$50 from spot
ZOMMA_WEIGHT = 0.6
VOMMA_WEIGHT = 0.4

# Use df_greeks from earlier (has vomma_exp, zomma_exp, gamma)
df_car = df_greeks.copy()

# Calculate time to expiry in years for each trade
# For 0DTE, we need to calculate TTE from trade timestamp to 4:00 PM ET
market_close = df_car['timestamp'].dt.normalize() + pd.Timedelta(hours=16)
market_close = market_close.dt.tz_localize(ET) if df_car['timestamp'].dt.tz is None else market_close.dt.tz_convert(ET)
df_car['tte_hours'] = (market_close - df_car['timestamp'].dt.tz_convert(ET)).dt.total_seconds() / 3600
df_car['tte_hours'] = df_car['tte_hours'].clip(lower=0.1)  # Minimum 6 minutes to avoid division issues
df_car['tte_years'] = df_car['tte_hours'] / (252 * 6.5)  # Trading hours per year

# Time decay amplifier: τ^(-0.5), capped at 30
df_car['time_amplifier'] = np.clip(1 / np.sqrt(df_car['tte_years']), 1, 30)

# Calculate GEX for each trade (needed for net gamma sign)
# GEX = gamma * size * spot² * 0.01, side-weighted
if 'gamma' not in df_car.columns:
    greeks_car = calculate_greeks(
        spot=df_car['spot'].values,
        strike=df_car['strike'].values,
        tte=df_car['tte_years'].values,
        rate=0.05,
        price=df_car['price'].values,
        is_call=(df_car['opt_type'] == 'C').values
    )
    df_car['gamma'] = greeks_car['gamma']

df_car['gex_raw'] = df_car['gamma'] * df_car['size'] * (df_car['spot'] ** 2) * 0.01
df_car.loc[df_car['trade_dir'] == 'SELL', 'gex_raw'] *= -1

# Per-trade CAR contribution (side-weighted)
# Positive CAR = risk of downside acceleration
df_car['car_contribution'] = (
    ZOMMA_WEIGHT * df_car['zomma_exp'] + 
    VOMMA_WEIGHT * df_car['vomma_exp']
) * df_car['time_amplifier']

# Create intervals
df_car['interval'] = df_car['timestamp'].dt.floor(f'{CAR_INTERVAL_MINUTES}min')

# Filter to strike range
spot = result.spot
df_car_filtered = df_car[
    (df_car['strike'] >= spot - CAR_STRIKE_RANGE) &
    (df_car['strike'] <= spot + CAR_STRIKE_RANGE)
].copy()

# Aggregate by interval
car_by_interval = df_car_filtered.groupby('interval').agg({
    'car_contribution': 'sum',
    'zomma_exp': 'sum',
    'vomma_exp': 'sum',
    'gex_raw': 'sum',
    'spot': 'last',
    'tte_hours': 'mean',
    'time_amplifier': 'mean'
}).reset_index()

# Net gamma by interval (from GEX - negative GEX means short gamma for dealers)
car_by_interval['net_gamma_sign'] = np.where(car_by_interval['gex_raw'] < 0, -1, 1)

# Aggregate CAR score = -Γ_sign × (weighted sum of positive zomma/vomma contributions)
# Only count positive zomma/vomma (the risky regime)
car_by_interval['zomma_positive'] = car_by_interval['zomma_exp'].clip(lower=0)
car_by_interval['vomma_positive'] = car_by_interval['vomma_exp'].clip(lower=0)

car_by_interval['car_score'] = -car_by_interval['net_gamma_sign'] * (
    ZOMMA_WEIGHT * car_by_interval['zomma_positive'] + 
    VOMMA_WEIGHT * car_by_interval['vomma_positive']
) * car_by_interval['time_amplifier'] / 1e6  # Scale to millions

# Convert to ET
car_by_interval['interval_dt'] = car_by_interval['interval'].dt.tz_convert(ET)

# Aggregate by interval AND strike for bubble chart
car_by_strike_interval = df_car_filtered.groupby(['interval', 'strike']).agg({
    'car_contribution': 'sum',
    'zomma_exp': 'sum',
    'vomma_exp': 'sum',
    'gex_raw': 'sum',
    'spot': 'last',
    'size': 'sum',
    'time_amplifier': 'mean'
}).reset_index()

# Calculate per-strike CAR (only when conditions are risky)
car_by_strike_interval['zomma_pos'] = car_by_strike_interval['zomma_exp'].clip(lower=0)
car_by_strike_interval['vomma_pos'] = car_by_strike_interval['vomma_exp'].clip(lower=0)

# Per-strike net gamma sign
car_by_strike_interval['gamma_sign'] = np.where(car_by_strike_interval['gex_raw'] < 0, -1, 1)

# Strike CAR = -gamma_sign × weighted Greeks × time_amplifier
car_by_strike_interval['strike_car'] = -car_by_strike_interval['gamma_sign'] * (
    ZOMMA_WEIGHT * car_by_strike_interval['zomma_pos'] + 
    VOMMA_WEIGHT * car_by_strike_interval['vomma_pos']
) * car_by_strike_interval['time_amplifier']

car_by_strike_interval['interval_dt'] = car_by_strike_interval['interval'].dt.tz_convert(ET)

print(f"CAR Analysis: {len(car_by_interval)} intervals, {len(car_by_strike_interval)} strike-interval combinations")
print(f"Time range: {car_by_interval['interval_dt'].min()} to {car_by_interval['interval_dt'].max()}")
print(f"Net GEX: ${car_by_interval['gex_raw'].sum()/1e6:.2f}M ({'SHORT GAMMA' if car_by_interval['gex_raw'].sum() < 0 else 'LONG GAMMA'})")

if len(car_by_interval) > 0:
    # Create figure with 3 panels
    if DARK_THEME:
        plt.style.use("dark_background")
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(20, 22), height_ratios=[1.5, 2, 1],
                                         gridspec_kw={"hspace": 0.15})
    
    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2, ax3]:
            ax.set_facecolor("#0a1628")
    
    times = car_by_interval['interval_dt'].values
    car_scores = car_by_interval['car_score'].values
    
    # ========== PANEL 1: CAR MAGNITUDE OVER TIME ==========
    # Fill area under curve with gradient based on magnitude
    # Positive CAR = danger (red gradient), Negative = safe (green gradient)
    
    ax1.axhline(y=0, color='#4a5568', linestyle='-', linewidth=1, alpha=0.5)
    
    # Create filled areas with color intensity based on magnitude
    positive_car = np.maximum(car_scores, 0)
    negative_car = np.minimum(car_scores, 0)
    
    # Fill positive (danger zone) with red gradient
    ax1.fill_between(times, 0, positive_car, alpha=0.4, color='#ef4444', label='Downside Acceleration Risk')
    ax1.fill_between(times, 0, negative_car, alpha=0.4, color='#22c55e', label='Dampening Effect')
    
    # Plot the CAR line
    ax1.plot(times, car_scores, color='white', linewidth=2, alpha=0.9)
    
    # Add magnitude markers at peaks
    peak_idx = np.argmax(np.abs(car_scores))
    peak_val = car_scores[peak_idx]
    peak_time = times[peak_idx]
    ax1.scatter([peak_time], [peak_val], s=100, c='#fbbf24', edgecolors='white', 
                linewidths=2, zorder=10, label=f'Peak: {peak_val:.1f}')
    ax1.annotate(f'{peak_val:.1f}', xy=(peak_time, peak_val), 
                 xytext=(10, 10 if peak_val > 0 else -20), textcoords='offset points',
                 fontsize=11, color='white', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#fbbf24', alpha=0.8))
    
    # Underlying price on secondary axis
    ax1_price = ax1.twinx()
    spot_prices = car_by_interval['spot'].values
    ax1_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.6, linestyle='--')
    ax1_price.set_ylabel('Spot Price ($)', fontsize=11, color='#fbbf24')
    ax1_price.tick_params(axis='y', colors='#fbbf24')
    
    ax1.set_ylabel('CAR Score (Millions)', fontsize=12, color='white')
    net_car = car_scores.sum()
    avg_car = car_scores.mean()
    max_car = car_scores.max()
    min_car = car_scores.min()
    
    # Determine overall risk assessment
    if max_car > 5:
        risk_level = "EXTREME RISK"
        risk_color = "#ef4444"
    elif max_car > 2.5:
        risk_level = "HIGH RISK"
        risk_color = "#f97316"
    elif max_car > 1:
        risk_level = "ELEVATED RISK"
        risk_color = "#eab308"
    else:
        risk_level = "NORMAL"
        risk_color = "#22c55e"
    
    ax1.set_title(f"Convexity Acceleration Risk (CAR) Over Time\n"
                  f"Peak: {max_car:.2f} | Min: {min_car:.2f} | Avg: {avg_car:.2f} | Assessment: {risk_level}",
                  fontsize=14, fontweight='bold', color='white', pad=10)
    
    # Add risk level indicator box
    ax1.text(0.02, 0.95, risk_level, transform=ax1.transAxes, fontsize=14, fontweight='bold',
             verticalalignment='top', color=risk_color,
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a2744', edgecolor=risk_color, linewidth=2))
    
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax1.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30], tz=ET))
    ax1.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
    ax1.grid(axis='both', alpha=0.2, color='#4a5568')
    ax1.tick_params(colors='white')
    for spine in ax1.spines.values():
        spine.set_color('#4a5568')
    
    # ========== PANEL 2: STRIKE-LEVEL CAR CONTRIBUTION BUBBLE CHART ==========
    # Shows which strikes contribute most to CAR over time
    
    max_strike_car = car_by_strike_interval['strike_car'].abs().max()
    min_size = 5
    max_size = 400
    
    # Plot bubbles for each strike-interval combination
    for _, row in car_by_strike_interval.iterrows():
        x = row['interval_dt']
        y = row['strike']
        car_val = row['strike_car']
        
        if abs(car_val) < max_strike_car * 0.01:  # Skip tiny contributions
            continue
        
        size = min_size + (abs(car_val) / max_strike_car) * (max_size - min_size) if max_strike_car > 0 else min_size
        
        # Color: red for positive (risk), green for negative (dampening)
        # Alpha varies with magnitude
        if car_val > 0:
            color = '#ef4444'
            alpha = min(0.9, 0.2 + 0.7 * (car_val / max_strike_car)) if max_strike_car > 0 else 0.5
        else:
            color = '#22c55e'
            alpha = min(0.9, 0.2 + 0.7 * (abs(car_val) / max_strike_car)) if max_strike_car > 0 else 0.5
        
        ax2.scatter(x, y, s=size, c=color, alpha=alpha, edgecolors='white', linewidths=0.3)
    
    # Underlying price line
    spot_by_interval = car_by_strike_interval.groupby('interval_dt')['spot'].mean().dropna()
    if len(spot_by_interval) > 0:
        ax2.plot(spot_by_interval.index, spot_by_interval.values,
                 color='#fbbf24', linewidth=2.5, alpha=0.9, label='Underlying Price', zorder=10)
    
    # Current spot line
    ax2.axhline(y=spot, color='#fbbf24', linewidth=2, linestyle='--', alpha=0.6,
                label=f'Current Spot: ${spot:,.0f}', zorder=9)
    
    ax2.set_ylabel('Strike Price', fontsize=12, color='white')
    ax2.set_title(f"Strike-Level CAR Contribution Over Time\n"
                  f"Red = Acceleration Risk (Short Γ + Zomma+ + Vomma+) | Green = Dampening (Long Γ)",
                  fontsize=14, fontweight='bold', color='white', pad=10)
    
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax2.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30], tz=ET))
    
    # Y-axis ticks
    all_strikes = sorted(car_by_strike_interval['strike'].unique())
    if len(all_strikes) > 0:
        strike_range = [min(all_strikes), max(all_strikes)]
        y_ticks = np.arange(int(strike_range[0] // 5) * 5, int(strike_range[1] // 5 + 1) * 5, 5)
        ax2.set_yticks(y_ticks)
    
    ax2.grid(axis='both', alpha=0.2, color='#4a5568')
    ax2.tick_params(colors='white')
    for spine in ax2.spines.values():
        spine.set_color('#4a5568')
    
    # Legend for bubble chart
    bubble_legend = [
        plt.scatter([], [], s=50, c='#ef4444', alpha=0.7, edgecolors='white', label='Small Risk'),
        plt.scatter([], [], s=200, c='#ef4444', alpha=0.7, edgecolors='white', label='Large Risk'),
        plt.scatter([], [], s=50, c='#22c55e', alpha=0.7, edgecolors='white', label='Small Dampening'),
        plt.scatter([], [], s=200, c='#22c55e', alpha=0.7, edgecolors='white', label='Large Dampening'),
        plt.Line2D([0], [0], color='#fbbf24', linewidth=2.5, label='Spot Price'),
    ]
    ax2.legend(handles=bubble_legend, loc='upper left', fontsize=9, framealpha=0.8,
               facecolor='#1a2744', title='CAR Contribution')
    
    # ========== PANEL 3: TOP CONTRIBUTING STRIKES ==========
    # Bar chart showing which strikes contribute most to CAR (aggregated)
    
    car_by_strike_total = car_by_strike_interval.groupby('strike').agg({
        'strike_car': 'sum',
        'zomma_pos': 'sum',
        'vomma_pos': 'sum',
        'size': 'sum'
    }).reset_index()
    
    # Sort by absolute CAR contribution
    car_by_strike_total['abs_car'] = car_by_strike_total['strike_car'].abs()
    car_by_strike_total = car_by_strike_total.sort_values('abs_car', ascending=False).head(20)
    
    strikes_top = car_by_strike_total['strike'].values
    car_values = car_by_strike_total['strike_car'].values / 1000  # Scale to thousands
    
    bar_colors = ['#ef4444' if c > 0 else '#22c55e' for c in car_values]
    bars = ax3.barh(range(len(strikes_top)), car_values, color=bar_colors, alpha=0.8)
    
    ax3.set_yticks(range(len(strikes_top)))
    ax3.set_yticklabels([f'${s:,.0f}' for s in strikes_top], fontsize=10)
    ax3.axvline(x=0, color='#4a5568', linestyle='-', linewidth=1)
    
    # Mark spot price
    spot_idx = np.argmin(np.abs(strikes_top - spot))
    if abs(strikes_top[spot_idx] - spot) < 10:
        ax3.get_yticklabels()[spot_idx].set_color('#fbbf24')
        ax3.get_yticklabels()[spot_idx].set_fontweight('bold')
    
    ax3.set_xlabel('Total CAR Contribution (K)', fontsize=12, color='white')
    ax3.set_title(f"Top 20 Contributing Strikes (Total Session)\n"
                  f"Red = Downside Risk | Green = Stabilizing",
                  fontsize=12, fontweight='bold', color='white', pad=10)
    ax3.tick_params(colors='white')
    ax3.grid(axis='x', alpha=0.2, color='#4a5568')
    for spine in ax3.spines.values():
        spine.set_color('#4a5568')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, car_values)):
        label_x = val + (max(car_values) - min(car_values)) * 0.02 if val >= 0 else val - (max(car_values) - min(car_values)) * 0.02
        ha = 'left' if val >= 0 else 'right'
        ax3.text(label_x, i, f'{val:.1f}K', va='center', ha=ha, fontsize=9, color='white')
    
    fig.suptitle(f"Convexity Acceleration Risk Analysis — {TRADE_DATE} | {TIME_LABEL}\n"
                 f"CAR = -Γ_sign × (0.6·Zomma+ + 0.4·Vomma+) × τ^(-0.5) | {MONEYNESS_LABEL}",
                 fontsize=16, color="white", fontweight="bold", y=0.995)
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()
    
    # Detailed Summary
    print(f"\n{'='*70}")
    print(f"CONVEXITY ACCELERATION RISK (CAR) SUMMARY")
    print(f"{'='*70}")
    print(f"\nAGGREGATE METRICS:")
    print(f"  Peak CAR Score:    {max_car:.2f}")
    print(f"  Min CAR Score:     {min_car:.2f}")
    print(f"  Average CAR:       {avg_car:.2f}")
    print(f"  Risk Assessment:   {risk_level}")
    
    # Danger zone analysis
    danger_intervals = car_by_interval[car_by_interval['car_score'] > 2.5]
    if len(danger_intervals) > 0:
        print(f"\n  ⚠️  DANGER ZONE INTERVALS: {len(danger_intervals)} ({len(danger_intervals)/len(car_by_interval)*100:.1f}%)")
        print(f"      Time range: {danger_intervals['interval_dt'].min().strftime('%H:%M')} - {danger_intervals['interval_dt'].max().strftime('%H:%M')} ET")
        print(f"      Peak danger: {danger_intervals['car_score'].max():.2f} at {danger_intervals.loc[danger_intervals['car_score'].idxmax(), 'interval_dt'].strftime('%H:%M')} ET")
    else:
        print(f"\n  ✓ No danger zone intervals detected (CAR < 2.5)")
    
    print(f"\nCOMPONENT BREAKDOWN:")
    total_zomma_contrib = car_by_interval['zomma_positive'].sum() * ZOMMA_WEIGHT
    total_vomma_contrib = car_by_interval['vomma_positive'].sum() * VOMMA_WEIGHT
    print(f"  Zomma Contribution: {total_zomma_contrib/1e6:.2f}M ({ZOMMA_WEIGHT*100:.0f}% weight)")
    print(f"  Vomma Contribution: {total_vomma_contrib/1e6:.2f}M ({VOMMA_WEIGHT*100:.0f}% weight)")
    print(f"  Net GEX: ${car_by_interval['gex_raw'].sum()/1e6:.2f}M")
    
    print(f"\nTOP RISK STRIKES (Positive CAR):")
    top_risk = car_by_strike_total[car_by_strike_total['strike_car'] > 0].head(5)
    for _, row in top_risk.iterrows():
        print(f"  ${row['strike']:,.0f}: {row['strike_car']/1000:.1f}K CAR")
    
    print(f"\nTOP STABILIZING STRIKES (Negative CAR):")
    top_stable = car_by_strike_total[car_by_strike_total['strike_car'] < 0].sort_values('strike_car').head(5)
    for _, row in top_stable.iterrows():
        print(f"  ${row['strike']:,.0f}: {row['strike_car']/1000:.1f}K CAR")
    
    print(f"\n{'='*70}")
else:
    print("Insufficient data for CAR analysis")

In [ ]:
# ============================================================
# 4e. CAR FORMULA COMPARISON - Directional Risk Analysis
# ============================================================
# KEY INSIGHT: Trade side determines whether flow is DESTABILIZING or STABILIZING
#
# Risk Contribution = how dealer hedging pressure changes with vol spike:
#   - Customer BUY + zomma > 0: Dealer goes MORE short gamma → DESTABILIZING
#   - Customer BUY + zomma < 0: Dealer goes LESS short gamma → STABILIZING  
#   - Customer SELL + zomma > 0: Dealer goes MORE long gamma → STABILIZING
#   - Customer SELL + zomma < 0: Dealer goes LESS long gamma → DESTABILIZING
#
# The current zomma_exp already has correct sign:
#   - BUY: zomma_exp = +zomma (positive = destabilizing)
#   - SELL: zomma_exp = -zomma (positive zomma → negative exp = stabilizing)
#
# CAR Metrics:
#   - CAR_net: Sum of all contributions (directional signal)
#   - CAR_gross: Sum of |contributions| (total activity)
#   - CAR_destabilizing: Sum of positive contributions only
#   - CAR_stabilizing: Sum of |negative contributions|
#   - CAR_ratio: destabilizing / stabilizing (>1 = risk building)

import warnings
from scipy.stats import norm
warnings.filterwarnings('ignore')

# Calculate Charm
def calculate_charm(spot, strike, tte, rate, sigma, is_call):
    tte = np.maximum(tte, 1e-8)
    sigma = np.maximum(sigma, 0.01)
    d1 = (np.log(spot / strike) + (rate + 0.5 * sigma**2) * tte) / (sigma * np.sqrt(tte))
    d2 = d1 - sigma * np.sqrt(tte)
    phi_d1 = norm.pdf(d1)
    sqrt_tte = np.sqrt(tte)
    charm_base = -phi_d1 * (2 * rate * tte - d2 * sigma * sqrt_tte) / (2 * tte * sigma * sqrt_tte)
    return np.where(is_call, charm_base, -charm_base)

df_comp = df_car.copy()

if 'iv' not in df_comp.columns:
    df_comp['iv'] = 0.20

df_comp['charm'] = calculate_charm(
    spot=df_comp['spot'].values,
    strike=df_comp['strike'].values,
    tte=df_comp['tte_years'].values,
    rate=0.05,
    sigma=df_comp['iv'].values,
    is_call=(df_comp['opt_type'] == 'C').values
)

# Charm exposure (side-weighted like zomma_exp)
df_comp['charm_exp'] = df_comp['charm'] * df_comp['size']
df_comp.loc[df_comp['trade_dir'] == 'SELL', 'charm_exp'] *= -1

# Risk contribution is zomma_exp (already correctly signed!)
# Positive = destabilizing, Negative = stabilizing
df_comp['risk_contribution'] = df_comp['zomma_exp']

# Track direction for analysis
df_comp['is_destabilizing'] = df_comp['risk_contribution'] > 0

df_comp['interval'] = df_comp['timestamp'].dt.floor(f'{CAR_INTERVAL_MINUTES}min')

# Filter to strike range
spot = result.spot
df_comp_filtered = df_comp[
    (df_comp['strike'] >= spot - CAR_STRIKE_RANGE) &
    (df_comp['strike'] <= spot + CAR_STRIKE_RANGE)
].copy()

# Diagnostic: Trade-level breakdown
print("=== DIAGNOSTIC: Trade-Level Risk Contributions ===")
print(f"Total trades: {len(df_comp_filtered)}")

# Breakdown by trade direction and zomma sign
buy_trades = df_comp_filtered[df_comp_filtered['trade_dir'] == 'BUY']
sell_trades = df_comp_filtered[df_comp_filtered['trade_dir'] == 'SELL']

print(f"\nBUY trades: {len(buy_trades)}")
print(f"  Destabilizing (zomma > 0): {(buy_trades['zomma'] > 0).sum()}")
print(f"  Stabilizing (zomma < 0):   {(buy_trades['zomma'] < 0).sum()}")

print(f"\nSELL trades: {len(sell_trades)}")
print(f"  Stabilizing (zomma > 0):   {(sell_trades['zomma'] > 0).sum()}")
print(f"  Destabilizing (zomma < 0): {(sell_trades['zomma'] < 0).sum()}")

# Aggregate with DIRECTIONAL metrics
comp_by_interval = df_comp_filtered.groupby('interval').agg({
    'risk_contribution': 'sum',                    # CAR_net (signed)
    'zomma_exp': lambda x: x.abs().sum(),         # CAR_gross (absolute)
    'vomma_exp': 'sum',
    'charm_exp': 'sum',
    'gex_raw': 'sum',
    'spot': 'last',
    'time_amplifier': 'mean',
    'is_destabilizing': 'sum',                    # Count of destabilizing trades
    'size': 'count'                               # Total trades
}).reset_index()

comp_by_interval.rename(columns={
    'risk_contribution': 'car_net',
    'zomma_exp': 'car_gross',
    'is_destabilizing': 'destab_count',
    'size': 'trade_count'
}, inplace=True)

# Calculate destabilizing and stabilizing separately
destab_by_interval = df_comp_filtered[df_comp_filtered['risk_contribution'] > 0].groupby('interval')['risk_contribution'].sum()
stab_by_interval = df_comp_filtered[df_comp_filtered['risk_contribution'] < 0].groupby('interval')['risk_contribution'].sum().abs()

comp_by_interval['car_destabilizing'] = comp_by_interval['interval'].map(destab_by_interval).fillna(0)
comp_by_interval['car_stabilizing'] = comp_by_interval['interval'].map(stab_by_interval).fillna(0)
comp_by_interval['car_ratio'] = comp_by_interval['car_destabilizing'] / comp_by_interval['car_stabilizing'].replace(0, np.nan)

# Net gamma sign
comp_by_interval['gamma_sign'] = np.where(comp_by_interval['gex_raw'] < 0, -1, 1)

# Scale CAR metrics
scale = 1e6
comp_by_interval['car_net_scaled'] = comp_by_interval['car_net'] * comp_by_interval['time_amplifier'] / scale
comp_by_interval['car_gross_scaled'] = comp_by_interval['car_gross'] * comp_by_interval['time_amplifier'] / scale
comp_by_interval['car_destab_scaled'] = comp_by_interval['car_destabilizing'] * comp_by_interval['time_amplifier'] / scale
comp_by_interval['car_stab_scaled'] = comp_by_interval['car_stabilizing'] * comp_by_interval['time_amplifier'] / scale

# Charm Risk Indicator - DIRECTIONAL (like CAR)
# charm_exp already side-weighted: positive = destabilizing, negative = stabilizing

# Separate destabilizing and stabilizing charm flow
charm_destab_by_interval = df_comp_filtered[df_comp_filtered['charm_exp'] > 0].groupby('interval')['charm_exp'].sum()
charm_stab_by_interval = df_comp_filtered[df_comp_filtered['charm_exp'] < 0].groupby('interval')['charm_exp'].sum().abs()

comp_by_interval['cri_net'] = comp_by_interval['charm_exp']  # Already aggregated sum (signed)
comp_by_interval['cri_destabilizing'] = comp_by_interval['interval'].map(charm_destab_by_interval).fillna(0)
comp_by_interval['cri_stabilizing'] = comp_by_interval['interval'].map(charm_stab_by_interval).fillna(0)
comp_by_interval['cri_ratio'] = comp_by_interval['cri_destabilizing'] / comp_by_interval['cri_stabilizing'].replace(0, np.nan)

# Scale CRI metrics  
comp_by_interval['cri_net_scaled'] = comp_by_interval['cri_net'] * comp_by_interval['time_amplifier'] / scale
comp_by_interval['cri_destab_scaled'] = comp_by_interval['cri_destabilizing'] * comp_by_interval['time_amplifier'] / scale
comp_by_interval['cri_stab_scaled'] = comp_by_interval['cri_stabilizing'] * comp_by_interval['time_amplifier'] / scale

# Keep cri for backwards compatibility
comp_by_interval['cri'] = comp_by_interval['cri_net_scaled']

comp_by_interval['interval_dt'] = comp_by_interval['interval'].dt.tz_convert(ET)

print(f"\n=== Aggregated Interval Stats ===")
print(f"Intervals: {len(comp_by_interval)}")
print(f"Time range: {comp_by_interval['interval_dt'].min().strftime('%H:%M')} to {comp_by_interval['interval_dt'].max().strftime('%H:%M')} ET")

# ====== VISUALIZATION ======

if DARK_THEME:
    plt.style.use("dark_background")

fig = plt.figure(figsize=(22, 18))
fig.patch.set_facecolor("#0a1628")

# Create GridSpec for layout
gs = fig.add_gridspec(3, 2, height_ratios=[1.5, 1.5, 1], hspace=0.25, wspace=0.15)

times = comp_by_interval['interval_dt'].values
spot_prices = comp_by_interval['spot'].values

# ===== PANEL 1: CAR NET (Directional) =====
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor("#0a1628")

net_vals = comp_by_interval['car_net_scaled'].values
pos_net = np.maximum(net_vals, 0)
neg_net = np.minimum(net_vals, 0)

ax1.axhline(y=0, color='#4a5568', linestyle='-', linewidth=1, alpha=0.5)
ax1.fill_between(times, 0, pos_net, alpha=0.5, color='#ef4444', label='Destabilizing')
ax1.fill_between(times, 0, neg_net, alpha=0.5, color='#22c55e', label='Stabilizing')
ax1.plot(times, net_vals, color='white', linewidth=2, alpha=0.9)

# Peak annotation
if len(net_vals) > 0:
    peak_idx = np.argmax(np.abs(net_vals))
    ax1.scatter([times[peak_idx]], [net_vals[peak_idx]], s=100, c='#fbbf24', 
                edgecolors='white', linewidths=2, zorder=10)
    ax1.annotate(f'{net_vals[peak_idx]:.2f}', xy=(times[peak_idx], net_vals[peak_idx]),
                 xytext=(10, 10 if net_vals[peak_idx] > 0 else -20), textcoords='offset points',
                 fontsize=10, color='white', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#fbbf24', alpha=0.8))

# Spot overlay
ax1_price = ax1.twinx()
ax1_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.4, linestyle='--')
ax1_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax1_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax1.set_title(f'CAR NET (Directional)\n+ve = Destabilizing Flow | -ve = Stabilizing Flow\n'
              f'Peak: {net_vals.max():.2f} | Min: {net_vals.min():.2f}',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax1.set_ylabel('CAR Net (M)', fontsize=10, color='white')
ax1.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax1.grid(axis='both', alpha=0.2, color='#4a5568')
ax1.tick_params(colors='white')

# ===== PANEL 2: CAR GROSS (Total Activity) =====
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor("#0a1628")

gross_vals = comp_by_interval['car_gross_scaled'].values
ax2.fill_between(times, 0, gross_vals, alpha=0.5, color='#60a5fa')
ax2.plot(times, gross_vals, color='#60a5fa', linewidth=2, alpha=0.9)

ax2_price = ax2.twinx()
ax2_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.4, linestyle='--')
ax2_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax2_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax2.set_title(f'CAR GROSS (Total Vol-Gamma Activity)\n|Zomma| regardless of direction\n'
              f'Peak: {gross_vals.max():.2f} | Avg: {gross_vals.mean():.2f}',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax2.set_ylabel('CAR Gross (M)', fontsize=10, color='white')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax2.grid(axis='both', alpha=0.2, color='#4a5568')
ax2.tick_params(colors='white')

# ===== PANEL 3: DESTABILIZING vs STABILIZING STACKED =====
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor("#0a1628")

destab_vals = comp_by_interval['car_destab_scaled'].values
stab_vals = comp_by_interval['car_stab_scaled'].values

ax3.bar(times, destab_vals, width=0.003, color='#ef4444', alpha=0.7, label='Destabilizing')
ax3.bar(times, -stab_vals, width=0.003, color='#22c55e', alpha=0.7, label='Stabilizing')
ax3.axhline(y=0, color='white', linestyle='-', linewidth=1, alpha=0.5)

ax3_price = ax3.twinx()
ax3_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.6, linestyle='--')
ax3_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax3_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax3.set_title(f'Destabilizing vs Stabilizing Flow\nRed bars up = risk building | Green bars down = risk absorbing',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax3.set_ylabel('CAR Component (M)', fontsize=10, color='white')
ax3.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax3.grid(axis='both', alpha=0.2, color='#4a5568')
ax3.tick_params(colors='white')

# ===== PANEL 4: CAR RATIO =====
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor("#0a1628")

ratio_vals = comp_by_interval['car_ratio'].fillna(0).values
ratio_vals = np.clip(ratio_vals, 0, 10)  # Cap at 10 for visualization

# Color by ratio: green if <1 (more stabilizing), red if >1 (more destabilizing)
colors = ['#ef4444' if r > 1 else '#22c55e' for r in ratio_vals]
ax4.bar(times, ratio_vals, width=0.003, color=colors, alpha=0.7)
ax4.axhline(y=1, color='#fbbf24', linestyle='--', linewidth=2, alpha=0.8, label='Equilibrium (1:1)')

ax4.set_title(f'CAR Ratio (Destabilizing / Stabilizing)\n>1 = Net risk building | <1 = Net risk absorbing\n'
              f'Max: {ratio_vals.max():.2f} | Avg: {ratio_vals[ratio_vals > 0].mean():.2f}',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax4.set_ylabel('Ratio', fontsize=10, color='white')
ax4.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax4.grid(axis='both', alpha=0.2, color='#4a5568')
ax4.tick_params(colors='white')
ax4.set_ylim(0, min(10, ratio_vals.max() * 1.2) if ratio_vals.max() > 0 else 2)

# ===== PANEL 5: CHARM RISK INDICATOR (DIRECTIONAL) =====
ax5 = fig.add_subplot(gs[2, :])
ax5.set_facecolor("#0a1628")

cri_vals = comp_by_interval['cri_net_scaled'].values
pos_cri = np.maximum(cri_vals, 0)
neg_cri = np.minimum(cri_vals, 0)

ax5.axhline(y=0, color='#4a5568', linestyle='-', linewidth=1, alpha=0.5)
ax5.fill_between(times, 0, pos_cri, alpha=0.5, color='#ef4444', label='Destabilizing')
ax5.fill_between(times, 0, neg_cri, alpha=0.5, color='#22c55e', label='Stabilizing')
ax5.plot(times, cri_vals, color='#ec4899', linewidth=2, alpha=0.9)

# Peak annotation
if len(cri_vals) > 0 and not np.all(cri_vals == 0):
    peak_idx = np.argmax(np.abs(cri_vals))
    ax5.scatter([times[peak_idx]], [cri_vals[peak_idx]], s=100, c='#fbbf24', 
                edgecolors='white', linewidths=2, zorder=10)
    ax5.annotate(f'{cri_vals[peak_idx]:.2f}', xy=(times[peak_idx], cri_vals[peak_idx]),
                 xytext=(10, 10 if cri_vals[peak_idx] > 0 else -20), textcoords='offset points',
                 fontsize=10, color='white', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#fbbf24', alpha=0.8))

ax5_price = ax5.twinx()
ax5_price.plot(times, spot_prices, color='#fbbf24', linewidth=2, alpha=0.6, linestyle='--')
ax5_price.set_ylabel('Spot', fontsize=10, color='#fbbf24')
ax5_price.tick_params(axis='y', colors='#fbbf24')

cri_max = cri_vals.max() if len(cri_vals) > 0 else 0
cri_min = cri_vals.min() if len(cri_vals) > 0 else 0
total_cri_destab = comp_by_interval['cri_destab_scaled'].sum()
total_cri_stab = comp_by_interval['cri_stab_scaled'].sum()

ax5.set_title(f'Charm Risk Indicator (Time-Driven Convexity) — DIRECTIONAL\n'
              f'Peak: {cri_max:.2f} | Min: {cri_min:.2f} | Destab: {total_cri_destab:.1f}M | Stab: {total_cri_stab:.1f}M',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax5.set_xlabel('Time (ET)', fontsize=10, color='white')
ax5.set_ylabel('CRI Net (M)', fontsize=10, color='white')
ax5.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax5.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30], tz=ET))
ax5.grid(axis='both', alpha=0.2, color='#4a5568')
ax5.tick_params(colors='white')

fig.suptitle(f"Directional CAR Analysis — {TRADE_DATE} | {TIME_LABEL}\n"
             f"Trade Side Matters: BUY+zomma>0 = Destabilizing | SELL+zomma>0 = Stabilizing",
             fontsize=16, color="white", fontweight="bold", y=0.995)

fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
         fontsize=8, color="gray", transform=fig.transFigure)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ====== SUMMARY STATISTICS ======
# CRI Summary
total_cri_destab = comp_by_interval['cri_destab_scaled'].sum()
total_cri_stab = comp_by_interval['cri_stab_scaled'].sum()
total_cri_net = comp_by_interval['cri_net_scaled'].sum()

print(f"\nCHARM RISK INDICATOR (Time-Driven):")
print(f"  Total Destabilizing: {total_cri_destab:.2f}M")
print(f"  Total Stabilizing:   {total_cri_stab:.2f}M")
print(f"  Net CRI:             {total_cri_net:.2f}M ({'RISK BUILDING' if total_cri_net > 0 else 'RISK ABSORBING'})")
if total_cri_stab > 0:
    print(f"  Session Ratio:       {total_cri_destab/total_cri_stab:.2f}x")

print(f"\n{'='*80}")
print(f"DIRECTIONAL CAR ANALYSIS - SUMMARY")
print(f"{'='*80}")

total_destab = comp_by_interval['car_destab_scaled'].sum()
total_stab = comp_by_interval['car_stab_scaled'].sum()
total_net = comp_by_interval['car_net_scaled'].sum()

print(f"\nAGGREGATE FLOW:")
print(f"  Total Destabilizing: {total_destab:.2f}M")
print(f"  Total Stabilizing:   {total_stab:.2f}M")
print(f"  Net CAR:             {total_net:.2f}M ({'RISK BUILDING' if total_net > 0 else 'RISK ABSORBING'})")
print(f"  Session Ratio:       {total_destab/total_stab:.2f}x" if total_stab > 0 else "  Session Ratio: ∞")

# Find peak risk periods
high_risk_intervals = comp_by_interval[comp_by_interval['car_ratio'] > 2]
if len(high_risk_intervals) > 0:
    print(f"\nHIGH RISK INTERVALS (Ratio > 2:1):")
    for _, row in high_risk_intervals.iterrows():
        print(f"  {row['interval_dt'].strftime('%H:%M')} ET: Ratio {row['car_ratio']:.1f}x, Net {row['car_net_scaled']:.2f}M")
else:
    print(f"\n✓ No high-risk intervals (ratio > 2:1) detected")

# Correlation with spot movement
spot_changes = np.diff(spot_prices)
car_net_lagged = comp_by_interval['car_net_scaled'].values[:-1]
if len(spot_changes) > 5:
    corr = np.corrcoef(car_net_lagged, spot_changes)[0, 1]
    print(f"\nPREDICTIVE CORRELATION:")
    print(f"  CAR_net[t] vs Spot_change[t+1]: {corr:.3f}")
    if corr < -0.3:
        print(f"  ⚠️ Negative correlation suggests CAR may have predictive value for moves!")

# CRI Summary
total_cri_destab = comp_by_interval['cri_destab_scaled'].sum()
total_cri_stab = comp_by_interval['cri_stab_scaled'].sum()
total_cri_net = comp_by_interval['cri_net_scaled'].sum()

print(f"\nCHARM RISK INDICATOR (Time-Driven):")
print(f"  Total Destabilizing: {total_cri_destab:.2f}M")
print(f"  Total Stabilizing:   {total_cri_stab:.2f}M")
print(f"  Net CRI:             {total_cri_net:.2f}M ({'RISK BUILDING' if total_cri_net > 0 else 'RISK ABSORBING'})")
if total_cri_stab > 0:
    print(f"  Session Ratio:       {total_cri_destab/total_cri_stab:.2f}x")

print(f"\n{'='*80}")


In [ ]:
# ============================================================
# 4f. ADVANCED RISK METRICS - Gamma Distribution Analysis
# ============================================================
# These metrics address WHY negative CAR can precede large moves:
#
# The "Three-Stage Mechanism":
# 1. SETUP: Large negative CAR (aggregate shows stabilizing cushion)
# 2. TRIGGER: But gamma may be CONCENTRATED at specific strikes or FAR from spot
# 3. AMPLIFICATION: Move triggers gamma flip, dealers rush to re-hedge
#
# New Metrics:
# - Protective Gamma Ratio (PGR): Is gamma near spot? (Higher = more protection)
# - Gamma Distance Weighted (GDW): Proximity-weighted gamma exposure
# - CAR Acceleration: Rate of change predicts regime shifts better than level
# - Gamma Concentration Index (GCI): Herfindahl measure - concentration = fragility

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Use data from previous cell (df_comp, comp_by_interval)
# Recalculate per-interval strike-level data for distribution analysis

# ====== PARAMETERS ======
SIGMA_RANGE = 20  # Points for "near spot" definition (approximately 1 ATM straddle width)
DECAY_FACTOR = 20  # Exponential decay scale for GDW

# ====== PER-INTERVAL CALCULATIONS ======

results = []

for interval, group in df_comp_filtered.groupby('interval'):
    spot = group['spot'].iloc[-1]
    
    # Calculate gamma exposure for each strike
    # Using gex_raw if available, otherwise proxy from zomma * spot^2 / 100
    if 'gex_raw' in group.columns:
        strike_gamma = group.groupby('strike').agg({
            'gex_raw': 'sum',
            'zomma_exp': 'sum',
            'risk_contribution': 'sum'
        }).reset_index()
    else:
        # Proxy: gamma ≈ zomma for small moves
        strike_gamma = group.groupby('strike').agg({
            'zomma_exp': 'sum',
            'risk_contribution': 'sum'
        }).reset_index()
        strike_gamma['gex_raw'] = strike_gamma['zomma_exp']
    
    strikes = strike_gamma['strike'].values
    gamma_abs = strike_gamma['gex_raw'].abs().values
    gamma_total = gamma_abs.sum()
    
    if gamma_total == 0:
        results.append({
            'interval': interval,
            'pgr': np.nan,
            'gdw': np.nan,
            'gci': np.nan,
            'gamma_total': 0,
            'car_net': group['risk_contribution'].sum(),
            'spot': spot
        })
        continue
    
    # 1. Protective Gamma Ratio (PGR)
    # Gamma within SIGMA_RANGE of spot / total gamma
    near_spot_mask = np.abs(strikes - spot) <= SIGMA_RANGE
    gamma_near = gamma_abs[near_spot_mask].sum()
    pgr = gamma_near / gamma_total
    
    # 2. Gamma Distance Weighted (GDW)
    # Sum of gamma * exp(-|K-S| / DECAY_FACTOR)
    distances = np.abs(strikes - spot)
    weights = np.exp(-distances / DECAY_FACTOR)
    gdw = (gamma_abs * weights).sum()
    
    # 3. Gamma Concentration Index (GCI)
    # Herfindahl-Hirschman Index of gamma distribution
    # Sum of (gamma_i / gamma_total)^2
    gamma_shares = gamma_abs / gamma_total
    gci = (gamma_shares ** 2).sum()
    
    results.append({
        'interval': interval,
        'pgr': pgr,
        'gdw': gdw,
        'gci': gci,
        'gamma_total': gamma_total,
        'car_net': group['risk_contribution'].sum(),
        'spot': spot
    })

metrics_df = pd.DataFrame(results)
metrics_df['interval_dt'] = metrics_df['interval'].dt.tz_convert(ET)

# 4. CAR Acceleration
# Change in CAR_net from previous interval
metrics_df['car_acceleration'] = metrics_df['car_net'].diff()

# Merge with existing interval data
metrics_merged = comp_by_interval.merge(
    metrics_df[['interval', 'pgr', 'gdw', 'gci', 'car_acceleration']],
    on='interval',
    how='left'
)

# Scale GDW for visualization
gdw_scale = metrics_df['gdw'].abs().max() if metrics_df['gdw'].abs().max() > 0 else 1
metrics_df['gdw_scaled'] = metrics_df['gdw'] / gdw_scale

# ====== DIAGNOSTICS ======
print("=== ADVANCED RISK METRICS DIAGNOSTICS ===")
print(f"Intervals analyzed: {len(metrics_df)}")
print(f"SIGMA_RANGE for PGR: ±{SIGMA_RANGE} points")
print(f"DECAY_FACTOR for GDW: {DECAY_FACTOR} points")
print()
print(f"Protective Gamma Ratio (PGR):")
print(f"  Range: {metrics_df['pgr'].min():.2%} to {metrics_df['pgr'].max():.2%}")
print(f"  Mean:  {metrics_df['pgr'].mean():.2%}")
print(f"  Interpretation: Higher = more gamma near spot = better protection")
print()
print(f"Gamma Distance Weighted (GDW):")
print(f"  Range: {metrics_df['gdw'].min():.2e} to {metrics_df['gdw'].max():.2e}")
print(f"  Interpretation: Higher = more effective gamma (proximity-adjusted)")
print()
print(f"Gamma Concentration Index (GCI):")
print(f"  Range: {metrics_df['gci'].min():.4f} to {metrics_df['gci'].max():.4f}")
print(f"  Interpretation: Higher = more concentrated = more fragile")
print()
print(f"CAR Acceleration:")
print(f"  Range: {metrics_df['car_acceleration'].min():.2e} to {metrics_df['car_acceleration'].max():.2e}")
print(f"  Interpretation: Sharp changes signal regime transitions")

# ====== VISUALIZATION ======

if DARK_THEME:
    plt.style.use("dark_background")

fig = plt.figure(figsize=(22, 20))
fig.patch.set_facecolor("#0a1628")

gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 0.8], hspace=0.28, wspace=0.15)

times = metrics_df['interval_dt'].values
spot_prices = metrics_df['spot'].values

# ===== PANEL 1: Protective Gamma Ratio (PGR) =====
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor("#0a1628")

pgr_vals = metrics_df['pgr'].values * 100  # Convert to percentage

# Color by PGR level: green=high protection, red=low protection
colors = ['#22c55e' if p > 50 else '#fbbf24' if p > 25 else '#ef4444' for p in pgr_vals]
ax1.bar(times, pgr_vals, width=0.003, color=colors, alpha=0.7)
ax1.axhline(y=50, color='#22c55e', linestyle='--', linewidth=2, alpha=0.6, label='Strong (>50%)')
ax1.axhline(y=25, color='#fbbf24', linestyle='--', linewidth=2, alpha=0.6, label='Moderate (>25%)')

# Spot overlay
ax1_price = ax1.twinx()
ax1_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.4, linestyle='--')
ax1_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax1_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax1.set_title(f'Protective Gamma Ratio (PGR)\n'
              f'Γ within ±{SIGMA_RANGE} pts of spot / Γ total\n'
              f'Green=Strong Protection | Red=Vulnerable',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax1.set_ylabel('PGR (%)', fontsize=10, color='white')
ax1.set_ylim(0, 100)
ax1.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax1.grid(axis='both', alpha=0.2, color='#4a5568')
ax1.tick_params(colors='white')

# ===== PANEL 2: Gamma Distance Weighted (GDW) =====
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor("#0a1628")

gdw_vals = metrics_df['gdw'].values
gdw_normalized = metrics_df['gdw_scaled'].values

ax2.fill_between(times, 0, gdw_normalized, alpha=0.5, color='#60a5fa')
ax2.plot(times, gdw_normalized, color='#60a5fa', linewidth=2, alpha=0.9)

ax2_price = ax2.twinx()
ax2_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.4, linestyle='--')
ax2_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax2_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax2.set_title(f'Gamma Distance Weighted (GDW)\n'
              f'Σ Γᵢ × exp(-|Kᵢ-S|/{DECAY_FACTOR})\n'
              f'Higher = More Effective Gamma (proximity-adjusted)',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax2.set_ylabel('GDW (normalized)', fontsize=10, color='white')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax2.grid(axis='both', alpha=0.2, color='#4a5568')
ax2.tick_params(colors='white')

# ===== PANEL 3: Gamma Concentration Index (GCI) =====
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor("#0a1628")

gci_vals = metrics_df['gci'].values

# Color by concentration: red=concentrated/fragile, green=dispersed/stable
colors = ['#ef4444' if g > 0.3 else '#fbbf24' if g > 0.15 else '#22c55e' for g in gci_vals]
ax3.bar(times, gci_vals, width=0.003, color=colors, alpha=0.7)
ax3.axhline(y=0.30, color='#ef4444', linestyle='--', linewidth=2, alpha=0.6, label='High Concentration (>0.30)')
ax3.axhline(y=0.15, color='#fbbf24', linestyle='--', linewidth=2, alpha=0.6, label='Moderate (>0.15)')

ax3_price = ax3.twinx()
ax3_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.4, linestyle='--')
ax3_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax3_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax3.set_title(f'Gamma Concentration Index (GCI) — Herfindahl Measure\n'
              f'Σ(Γᵢ/Γ_total)² — Higher = More Concentrated = More Fragile\n'
              f'Red=Concentrated (fragile) | Green=Dispersed (stable)',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax3.set_ylabel('GCI', fontsize=10, color='white')
ax3.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax3.grid(axis='both', alpha=0.2, color='#4a5568')
ax3.tick_params(colors='white')

# ===== PANEL 4: CAR Acceleration =====
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor("#0a1628")

car_accel = metrics_df['car_acceleration'].values / 1e6  # Scale to millions
car_accel_filled = np.nan_to_num(car_accel, 0)

pos_accel = np.maximum(car_accel_filled, 0)
neg_accel = np.minimum(car_accel_filled, 0)

ax4.axhline(y=0, color='#4a5568', linestyle='-', linewidth=1, alpha=0.5)
ax4.fill_between(times, 0, pos_accel, alpha=0.5, color='#ef4444', label='Accelerating Risk')
ax4.fill_between(times, 0, neg_accel, alpha=0.5, color='#22c55e', label='Decelerating Risk')
ax4.plot(times, car_accel_filled, color='white', linewidth=2, alpha=0.9)

# Mark extreme accelerations
if len(car_accel_filled) > 0:
    threshold = np.percentile(np.abs(car_accel_filled), 95)
    extreme_mask = np.abs(car_accel_filled) > threshold
    if extreme_mask.any():
        ax4.scatter(times[extreme_mask], car_accel_filled[extreme_mask], 
                    s=100, c='#fbbf24', edgecolors='white', linewidths=2, zorder=10,
                    label='Extreme (95th pctl)')

ax4_price = ax4.twinx()
ax4_price.plot(times, spot_prices, color='#fbbf24', linewidth=1.5, alpha=0.4, linestyle='--')
ax4_price.set_ylabel('Spot', fontsize=9, color='#fbbf24', alpha=0.6)
ax4_price.tick_params(axis='y', colors='#fbbf24', labelsize=8)

ax4.set_title(f'CAR Acceleration (∂CAR/∂t)\n'
              f'Rate of change in convexity risk\n'
              f'Sharp spikes precede regime changes',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax4.set_ylabel('CAR Change (M)', fontsize=10, color='white')
ax4.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax4.grid(axis='both', alpha=0.2, color='#4a5568')
ax4.tick_params(colors='white')

# ===== PANEL 5: Combined Risk Score =====
ax5 = fig.add_subplot(gs[2, 0])
ax5.set_facecolor("#0a1628")

# Composite Risk Score: Low PGR + High GCI + Extreme CAR Accel = High Risk
# Normalize each to 0-1 scale
pgr_risk = 1 - metrics_df['pgr'].fillna(0).values  # Invert: low PGR = high risk
gci_risk = (metrics_df['gci'].fillna(0).values - metrics_df['gci'].min()) / (metrics_df['gci'].max() - metrics_df['gci'].min() + 1e-8)
accel_risk = (np.abs(car_accel_filled) - np.abs(car_accel_filled).min()) / (np.abs(car_accel_filled).max() - np.abs(car_accel_filled).min() + 1e-8)

# Composite: weighted average (PGR most important)
composite_risk = 0.5 * pgr_risk + 0.3 * gci_risk + 0.2 * accel_risk

# Color by composite risk
colors = ['#ef4444' if r > 0.7 else '#fbbf24' if r > 0.4 else '#22c55e' for r in composite_risk]
ax5.bar(times, composite_risk * 100, width=0.003, color=colors, alpha=0.7)
ax5.axhline(y=70, color='#ef4444', linestyle='--', linewidth=2, alpha=0.6, label='High Risk (>70)')
ax5.axhline(y=40, color='#fbbf24', linestyle='--', linewidth=2, alpha=0.6, label='Elevated (>40)')

ax5_price = ax5.twinx()
ax5_price.plot(times, spot_prices, color='#fbbf24', linewidth=2, alpha=0.6, linestyle='--')
ax5_price.set_ylabel('Spot', fontsize=10, color='#fbbf24')
ax5_price.tick_params(axis='y', colors='#fbbf24')

ax5.set_title(f'Composite Fragility Score\n'
              f'50% PGR + 30% GCI + 20% CAR Accel\n'
              f'Red=High Fragility | Green=Stable',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax5.set_ylabel('Risk Score (%)', fontsize=10, color='white')
ax5.set_ylim(0, 100)
ax5.legend(loc='upper right', fontsize=9, framealpha=0.8, facecolor='#1a2744')
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
ax5.grid(axis='both', alpha=0.2, color='#4a5568')
ax5.tick_params(colors='white')

# ===== PANEL 6: PGR vs CAR Net Scatter =====
ax6 = fig.add_subplot(gs[2, 1])
ax6.set_facecolor("#0a1628")

car_net_scaled = metrics_df['car_net'].values / 1e6
pgr_pct = metrics_df['pgr'].values * 100

# Size by absolute CAR
sizes = 50 + 200 * (np.abs(car_net_scaled) / (np.abs(car_net_scaled).max() + 1e-8))

# Color by time (earlier=blue, later=red)
time_norm = np.linspace(0, 1, len(times))

scatter = ax6.scatter(pgr_pct, car_net_scaled, c=time_norm, cmap='coolwarm', 
                       s=sizes, alpha=0.7, edgecolors='white', linewidths=0.5)

# Add quadrant labels
ax6.axhline(y=0, color='white', linestyle='-', linewidth=1, alpha=0.5)
ax6.axvline(x=50, color='white', linestyle='-', linewidth=1, alpha=0.5)

ax6.text(75, car_net_scaled.max() * 0.8, 'PROTECTED\nRISK BUILDING', 
         fontsize=9, color='#fbbf24', ha='center', fontweight='bold')
ax6.text(25, car_net_scaled.max() * 0.8, 'VULNERABLE\nRISK BUILDING', 
         fontsize=9, color='#ef4444', ha='center', fontweight='bold')
ax6.text(75, car_net_scaled.min() * 0.8, 'PROTECTED\nSTABILIZING', 
         fontsize=9, color='#22c55e', ha='center', fontweight='bold')
ax6.text(25, car_net_scaled.min() * 0.8, 'VULNERABLE\nSTABILIZING', 
         fontsize=9, color='#60a5fa', ha='center', fontweight='bold')

cbar = plt.colorbar(scatter, ax=ax6, label='Time (early→late)')
cbar.ax.yaxis.label.set_color('white')
cbar.ax.tick_params(colors='white')

ax6.set_title('PGR vs CAR Net (Phase Diagram)\n'
              'Quadrants reveal protection level during risk buildup',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax6.set_xlabel('PGR (%)', fontsize=10, color='white')
ax6.set_ylabel('CAR Net (M)', fontsize=10, color='white')
ax6.grid(axis='both', alpha=0.2, color='#4a5568')
ax6.tick_params(colors='white')

# ===== PANEL 7: Key Insight Summary =====
ax7 = fig.add_subplot(gs[3, :])
ax7.set_facecolor("#0a1628")
ax7.axis('off')

# Calculate key insights
avg_pgr = metrics_df['pgr'].mean() * 100
max_gci = metrics_df['gci'].max()
max_accel = np.abs(car_accel_filled).max()

# Find vulnerable periods: Low PGR + High CAR
vulnerable_mask = (metrics_df['pgr'] < 0.3) & (metrics_df['car_net'].abs() > metrics_df['car_net'].abs().median())
vulnerable_periods = metrics_df[vulnerable_mask]

# Session verdict
if avg_pgr < 30:
    pgr_verdict = "⚠️ LOW PROTECTION - Gamma concentrated away from spot"
elif avg_pgr < 50:
    pgr_verdict = "⚡ MODERATE PROTECTION - Some gamma near spot"
else:
    pgr_verdict = "✅ HIGH PROTECTION - Gamma well-distributed near spot"

if max_gci > 0.3:
    gci_verdict = "⚠️ HIGH CONCENTRATION - Few strikes dominate gamma"
elif max_gci > 0.15:
    gci_verdict = "⚡ MODERATE CONCENTRATION - Watch for strike flips"
else:
    gci_verdict = "✅ WELL DISPERSED - Gamma broadly distributed"

summary_text = f"""
SESSION GAMMA DISTRIBUTION ANALYSIS

PROTECTIVE GAMMA RATIO (PGR):
  Average: {avg_pgr:.1f}%  |  {pgr_verdict}
  
GAMMA CONCENTRATION INDEX (GCI):
  Maximum: {max_gci:.3f}  |  {gci_verdict}

CAR ACCELERATION:
  Maximum |∂CAR/∂t|: {max_accel:.2f}M  |  Sharp spikes signal regime changes

VULNERABLE PERIODS (Low PGR + High |CAR|):
  {len(vulnerable_periods)} intervals detected
"""

if len(vulnerable_periods) > 0:
    for _, row in vulnerable_periods.head(5).iterrows():
        summary_text += f"  • {row['interval_dt'].strftime('%H:%M')} ET: PGR={row['pgr']*100:.0f}%, CAR={row['car_net']/1e6:.1f}M\n"

summary_text += f"""
KEY INSIGHT:
When aggregate CAR shows "stabilizing" (negative) but PGR is LOW (<30%),
the cushion is illusory - gamma is far from spot and won't provide protection.
A move toward concentrated gamma strikes can trigger rapid dealer re-hedging.
"""

ax7.text(0.02, 0.95, summary_text, transform=ax7.transAxes, fontsize=11, color='white',
         fontfamily='monospace', verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a2744', edgecolor='#4a5568', alpha=0.9))

fig.suptitle(f"Advanced Gamma Distribution Metrics — {TRADE_DATE} | {TIME_LABEL}\n"
             f"Understanding WHY Aggregate CAR Can Mask Local Fragility",
             fontsize=16, color="white", fontweight="bold", y=0.995)

fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
         fontsize=8, color="gray", transform=fig.transFigure)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ====== PRINT DETAILED SUMMARY ======
print(f"\n{'='*80}")
print("ADVANCED GAMMA DISTRIBUTION ANALYSIS - SUMMARY")
print(f"{'='*80}")
print(summary_text)


In [ ]:
# ============================================================
# 5. GEX INTERVAL MAP (Bubble Chart)
# ============================================================
# Shows GEX evolution over time by strike
# - X-axis: Time intervals (configurable, default 5-min)
# - Y-axis: Strike price
# - Bubble size: GEX magnitude
# - Bubble color: Green (positive/stabilizing), Red (negative/destabilizing)
# - Yellow line: Underlying price evolution
# - Red dashed line: Current spot price

INTERVAL_MINUTES = 5
INTERVAL_STRIKE_RANGE = 50  # ±$50 from spot

# Prepare interval data
df_interval = df.copy()

# Calculate GEX for each trade: gamma * size * spot² * 0.01
# Side-weighted: flip sign for SELL trades
if 'gamma' not in df_interval.columns:
    greeks_int = calculate_greeks(
        spot=df_interval['spot'].values,
        strike=df_interval['strike'].values,
        tte=df_interval['tte_years'].values,
        rate=0.05,
        price=df_interval['price'].values,
        is_call=(df_interval['opt_type'] == 'C').values
    )
    df_interval['gamma'] = greeks_int['gamma']

df_interval['gex_raw'] = df_interval['gamma'] * df_interval['size'] * (df_interval['spot'] ** 2) * 0.01
df_interval.loc[df_interval['trade_dir'] == 'SELL', 'gex_raw'] *= -1

# Create time intervals
df_interval['interval'] = df_interval['timestamp'].dt.floor(f'{INTERVAL_MINUTES}min')

# Filter to strike range
spot = result.spot
df_interval = df_interval[
    (df_interval['strike'] >= spot - INTERVAL_STRIKE_RANGE) &
    (df_interval['strike'] <= spot + INTERVAL_STRIKE_RANGE)
].copy()

# Aggregate by interval and strike
interval_gex = df_interval.groupby(['interval', 'strike']).agg({
    'gex_raw': 'sum',
    'spot': 'last'
}).reset_index()
interval_gex.columns = ['interval_dt', 'strike', 'net_gex', 'avg_spot']

# Convert to ET for display
interval_gex['interval_dt'] = interval_gex['interval_dt'].dt.tz_convert(ET)

print(f"GEX Interval data: {len(interval_gex):,} interval-strike combinations")
print(f"Time range: {interval_gex['interval_dt'].min()} to {interval_gex['interval_dt'].max()}")

if len(interval_gex) > 0:
    # Calculate bubble sizes
    max_gex = interval_gex['net_gex'].abs().max()
    min_size = 10
    max_size = 300
    max_abs_gex = max(abs(interval_gex['net_gex'].min()), abs(interval_gex['net_gex'].max()))
    
    # Create figure
    if DARK_THEME:
        plt.style.use("dark_background")
    
    fig, ax = plt.subplots(figsize=(20, 14))
    
    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        ax.set_facecolor("#0a1628")
    
    # Plot bubbles
    for _, row in interval_gex.iterrows():
        x = row['interval_dt']
        y = row['strike']
        gex = row['net_gex']
        
        # Calculate size based on GEX magnitude
        size = min_size + (abs(gex) / max_gex) * (max_size - min_size) if max_gex > 0 else min_size
        
        # Color and alpha based on GEX sign and magnitude
        if gex > 0:
            color = '#22c55e'  # Green for stabilizing
            alpha = min(0.9, 0.3 + 0.6 * (gex / max_abs_gex)) if max_abs_gex > 0 else 0.5
        else:
            color = '#ef4444'  # Red for destabilizing
            alpha = min(0.9, 0.3 + 0.6 * (abs(gex) / max_abs_gex)) if max_abs_gex > 0 else 0.5
        
        ax.scatter(x, y, s=size, c=color, alpha=alpha, edgecolors='white', linewidths=0.3)
    
    # Underlying price line (historical spot evolution)
    spot_by_interval = interval_gex.groupby('interval_dt')['avg_spot'].mean().dropna()
    if len(spot_by_interval) > 0:
        ax.plot(
            spot_by_interval.index,
            spot_by_interval.values,
            color='#fbbf24',
            linewidth=2.5,
            alpha=0.9,
            label='Underlying Price',
            zorder=10
        )
    
    # Current spot price line (horizontal reference)
    trade_data_spot = spot_by_interval.iloc[-1] if len(spot_by_interval) > 0 else spot
    ax.axhline(
        y=spot,
        color='#ef4444',
        linewidth=2,
        linestyle='--',
        alpha=0.8,
        label=f'Current Spot: {spot:.0f}',
        zorder=9
    )
    
    # Styling
    ax.set_xlabel('Time (ET)', fontsize=12, color='white')
    ax.set_ylabel('Strike Price', fontsize=12, color='white')
    
    # Title
    ax.set_title(
        f"GEX Evolution Map — SPX 0DTE | {TRADE_DATE} | {TIME_LABEL}\n"
        f"{INTERVAL_MINUTES}-min intervals | ±${INTERVAL_STRIKE_RANGE} from spot | "
        f"{MONEYNESS_LABEL} | Green=Stabilizing Red=Destabilizing",
        fontsize=14, fontweight='bold', color='white', pad=15
    )
    
    # X-axis formatting
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=ET))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30], tz=ET))
    plt.xticks(rotation=45)
    
    # Y-axis: strike ticks
    all_strikes = sorted(interval_gex['strike'].unique())
    if len(all_strikes) > 0:
        strike_range = [min(all_strikes), max(all_strikes)]
        y_ticks = np.arange(int(strike_range[0] // 5) * 5, int(strike_range[1] // 5 + 1) * 5, 5)
        ax.set_yticks(y_ticks)
    
    ax.grid(axis='both', alpha=0.2, color='#4a5568')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#4a5568')
    
    # Legend
    legend_elements = [
        plt.scatter([], [], s=30, c='#22c55e', alpha=0.7, edgecolors='white', label='Small + GEX'),
        plt.scatter([], [], s=150, c='#22c55e', alpha=0.7, edgecolors='white', label='Large + GEX'),
        plt.scatter([], [], s=30, c='#ef4444', alpha=0.7, edgecolors='white', label='Small - GEX'),
        plt.scatter([], [], s=150, c='#ef4444', alpha=0.7, edgecolors='white', label='Large - GEX'),
        plt.Line2D([0], [0], color='#fbbf24', linewidth=2.5, label='Underlying Price'),
        plt.Line2D([0], [0], color='#ef4444', linewidth=2, linestyle='--', label=f'Current Spot: ${spot:,.0f}'),
    ]
    ax.legend(
        handles=legend_elements,
        loc='upper left',
        fontsize=9,
        framealpha=0.8,
        facecolor='#1a2744' if DARK_THEME else 'white',
        title='GEX Magnitude'
    )
    
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom",
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout()
    plt.show()
    
    # Summary stats
    total_positive = interval_gex[interval_gex['net_gex'] > 0]['net_gex'].sum()
    total_negative = interval_gex[interval_gex['net_gex'] < 0]['net_gex'].sum()
    net_total = total_positive + total_negative
    
    print(f"\nGEX Interval Summary:")
    print(f"  Total Positive GEX: ${total_positive/1e6:.2f}M (stabilizing)")
    print(f"  Total Negative GEX: ${total_negative/1e6:.2f}M (destabilizing)")
    print(f"  Net GEX: ${net_total/1e6:.2f}M ({'STABILIZING' if net_total > 0 else 'DESTABILIZING'})")
    print(f"  Intervals: {interval_gex['interval_dt'].nunique()}")
    print(f"  Strikes: {interval_gex['strike'].nunique()}")
else:
    print("No interval GEX data available")

In [ ]:
# ============================================================
# 5. VOLATILITY SKEW CHART
# ============================================================

# Calculate IV using Black-Scholes
df_iv = df.copy()
greeks = calculate_greeks(
    spot=df_iv['spot'].values,
    strike=df_iv['strike'].values,
    tte=df_iv['tte_years'].values,
    rate=0.05,
    price=df_iv['price'].values,
    is_call=(df_iv['opt_type'] == 'C').values
)
df_iv['iv'] = greeks['iv']

# Filter valid IV
df_iv = df_iv[(df_iv['iv'] > 0.05) & (df_iv['iv'] < 2.0)]

# Get spot from most recent trade with valid spot data
df_with_spot = df_iv[df_iv['spot'].notna()]
if len(df_with_spot) > 0:
    most_recent = df_with_spot.loc[df_with_spot['timestamp'].idxmax()]
    spot = most_recent['spot']
else:
    spot = df_iv['spot'].median()

df_iv = df_iv[(df_iv['strike'] >= spot - 100) & (df_iv['strike'] <= spot + 100)]

# Aggregate IV by strike and option type
iv_by_strike = df_iv.groupby(['strike', 'opt_type'])['iv'].mean().reset_index()
call_iv_df = iv_by_strike[iv_by_strike['opt_type'] == 'C'].sort_values('strike')
put_iv_df = iv_by_strike[iv_by_strike['opt_type'] == 'P'].sort_values('strike')

strikes = sorted(df_iv['strike'].unique())
call_iv = call_iv_df.set_index('strike')['iv'].reindex(strikes).values * 100
put_iv = put_iv_df.set_index('strike')['iv'].reindex(strikes).values * 100

# Calculate skew
valid_skew = []
valid_strikes = []
for i, s in enumerate(strikes):
    if not np.isnan(call_iv[i]) and not np.isnan(put_iv[i]):
        valid_skew.append(put_iv[i] - call_iv[i])
        valid_strikes.append(s)

if len(valid_strikes) > 0:
    # Create figure with 2 panels
    if DARK_THEME:
        plt.style.use("dark_background")
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), height_ratios=[2, 1], 
                                    gridspec_kw={"hspace": 0.15})
    
    if DARK_THEME:
        fig.patch.set_facecolor("#0a1628")
        for ax in [ax1, ax2]:
            ax.set_facecolor("#0a1628")
    
    # Top panel: IV curves with skew shading
    ax1.plot(call_iv_df['strike'], call_iv_df['iv'] * 100, color='#22c55e', linewidth=2, marker='o', markersize=4,
             label=f"Calls IV ({call_iv_df['iv'].mean()*100:.1f}% avg)")
    ax1.plot(put_iv_df['strike'], put_iv_df['iv'] * 100, color='#ef4444', linewidth=2, marker='s', markersize=4,
             label=f"Puts IV ({put_iv_df['iv'].mean()*100:.1f}% avg)")
    
    # Fill between for skew visualization
    common_strikes = sorted(set(call_iv_df['strike']) & set(put_iv_df['strike']))
    if len(common_strikes) > 0:
        c_iv_dict = dict(zip(call_iv_df['strike'], call_iv_df['iv'] * 100))
        p_iv_dict = dict(zip(put_iv_df['strike'], put_iv_df['iv'] * 100))
        c_vals = [c_iv_dict[s] for s in common_strikes]
        p_vals = [p_iv_dict[s] for s in common_strikes]
        ax1.fill_between(common_strikes, c_vals, p_vals, where=[p > c for p, c in zip(p_vals, c_vals)],
                         color='#ef4444', alpha=0.2, label='Put Skew')
        ax1.fill_between(common_strikes, c_vals, p_vals, where=[c > p for p, c in zip(p_vals, c_vals)],
                         color='#22c55e', alpha=0.2, label='Call Skew')
    
    # Spot line
    ax1.axvline(x=spot, color='#fbbf24', linestyle='--', linewidth=2, label=f"Spot ${spot:,.0f}")
    
    # ATM annotation
    atm_strike = min(strikes, key=lambda x: abs(x - spot))
    c_iv_dict = dict(zip(call_iv_df['strike'], call_iv_df['iv'] * 100))
    p_iv_dict = dict(zip(put_iv_df['strike'], put_iv_df['iv'] * 100))
    atm_call_iv = c_iv_dict.get(atm_strike, 0)
    atm_put_iv = p_iv_dict.get(atm_strike, 0)
    if atm_call_iv > 0 and atm_put_iv > 0:
        ax1.annotate(f"ATM: C={atm_call_iv:.1f}% P={atm_put_iv:.1f}%",
                     xy=(spot, max(atm_call_iv, atm_put_iv)),
                     xytext=(spot + 15, max(atm_call_iv, atm_put_iv) + 3),
                     fontsize=10, color='white',
                     bbox=dict(boxstyle='round', facecolor='#1a2744', alpha=0.8),
                     arrowprops=dict(arrowstyle='->', color='#fbbf24'))
    
    ax1.set_ylabel('Implied Volatility (%)', fontsize=12, color='white')
    ax1.set_title(f"Volatility Skew — {TRADE_DATE} | {TIME_LABEL}", color="white", fontsize=14, fontweight="bold")
    ax1.legend(loc='upper right', fontsize=10, framealpha=0.8, facecolor='#1a2744')
    ax1.tick_params(colors='white')
    for spine in ax1.spines.values(): spine.set_color('#4a5568')
    ax1.grid(axis='both', alpha=0.3, color='#4a5568')
    
    # Bottom panel: Skew (Put IV - Call IV)
    colors = ['#ef4444' if s > 0 else '#22c55e' for s in valid_skew]
    ax2.bar(valid_strikes, valid_skew, width=3, color=colors, alpha=0.8)
    ax2.axvline(x=spot, color='#fbbf24', linestyle='--', linewidth=2, alpha=0.7)
    ax2.axhline(y=0, color='#4a5568', linestyle='-', linewidth=1, alpha=0.5)
    
    ax2.set_xlabel('Strike', fontsize=12, color='white')
    ax2.set_ylabel('Put - Call IV (%)', fontsize=12, color='white')
    ax2.set_title('Volatility Skew (Put IV - Call IV)', color='white', fontsize=11)
    ax2.tick_params(colors='white')
    for spine in ax2.spines.values(): spine.set_color('#4a5568')
    ax2.grid(axis='y', alpha=0.3, color='#4a5568')
    
    fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom", 
             fontsize=8, color="gray", transform=fig.transFigure)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient IV data for skew chart")

In [ ]:
# ============================================================
# 6. TRADE SIDE DISTRIBUTION CHART
# ============================================================

# Create 2x2 panel figure
if DARK_THEME:
    plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.patch.set_facecolor("#0a1628" if DARK_THEME else "white")

# Side classification colors
side_colors = {
    'above_ask': '#16a34a',   # Dark green - aggressive buy
    'at_ask': '#22c55e',      # Green - buy
    'mid_market': '#3b82f6',  # Blue - neutral (classified as buy)
    'at_bid': '#ef4444',      # Red - sell  
    'below_bid': '#dc2626',   # Dark red - aggressive sell
}

# Panel 1: Trade side distribution (horizontal bar)
ax1 = axes[0, 0]
ax1.set_facecolor("#0a1628" if DARK_THEME else "white")

side_order = ['above_ask', 'at_ask', 'mid_market', 'at_bid', 'below_bid']
side_counts = df['side'].value_counts()
side_counts = side_counts.reindex([s for s in side_order if s in side_counts.index])

colors = [side_colors.get(s, '#888888') for s in side_counts.index]
bars = ax1.barh(side_counts.index, side_counts.values, color=colors, alpha=0.85)

# Add value labels
for bar, val in zip(bars, side_counts.values):
    pct = val / len(df) * 100
    ax1.text(val + side_counts.max() * 0.02, bar.get_y() + bar.get_height()/2, 
             f'{val:,} ({pct:.1f}%)', va='center', ha='left', color='white', fontsize=11)

ax1.set_xlabel('Trade Count', fontsize=12, color='white')
ax1.set_title('Trade Execution Side Distribution', color='white', fontsize=14, fontweight='bold')
ax1.tick_params(colors='white')
for spine in ax1.spines.values(): spine.set_color('#4a5568')
ax1.grid(axis='x', alpha=0.3, color='#4a5568')

# Panel 2: BUY vs SELL pie chart
ax2 = axes[0, 1]
ax2.set_facecolor("#0a1628" if DARK_THEME else "white")

dir_counts = df['trade_dir'].value_counts()
wedges, texts, autotexts = ax2.pie(
    dir_counts.values, 
    labels=dir_counts.index, 
    autopct='%1.1f%%',
    colors=['#22c55e', '#ef4444'],
    textprops={'color': 'white', 'fontsize': 14},
    explode=[0.02, 0.02],
    shadow=True,
    startangle=90
)
for autotext in autotexts:
    autotext.set_fontweight('bold')
    autotext.set_fontsize(16)

ax2.set_title('BUY vs SELL Classification\n(mid_market = BUY)', color='white', fontsize=14, fontweight='bold')

# Panel 3: By option type grouped bar
ax3 = axes[1, 0]
ax3.set_facecolor("#0a1628" if DARK_THEME else "white")

call_buys = len(df[(df['opt_type'] == 'C') & (df['trade_dir'] == 'BUY')])
call_sells = len(df[(df['opt_type'] == 'C') & (df['trade_dir'] == 'SELL')])
put_buys = len(df[(df['opt_type'] == 'P') & (df['trade_dir'] == 'BUY')])
put_sells = len(df[(df['opt_type'] == 'P') & (df['trade_dir'] == 'SELL')])

call_total = call_buys + call_sells
put_total = put_buys + put_sells
call_buy_pct = call_buys / call_total * 100 if call_total > 0 else 0
put_buy_pct = put_buys / put_total * 100 if put_total > 0 else 0

x = np.arange(2)
width = 0.35
bars1 = ax3.bar(x - width/2, [call_buys, put_buys], width, label='BUY', color='#22c55e', alpha=0.85)
bars2 = ax3.bar(x + width/2, [call_sells, put_sells], width, label='SELL', color='#ef4444', alpha=0.85)

ax3.set_xticks(x)
ax3.set_xticklabels(['CALLS', 'PUTS'], color='white', fontsize=14)
ax3.set_ylabel('Trade Count', fontsize=12, color='white')
ax3.set_title(f'Buy/Sell by Option Type\n(Call Buy: {call_buy_pct:.0f}% | Put Buy: {put_buy_pct:.0f}%)', 
              color='white', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', framealpha=0.8, facecolor='#1a2744', fontsize=12)
ax3.tick_params(colors='white')
for spine in ax3.spines.values(): spine.set_color('#4a5568')
ax3.grid(axis='y', alpha=0.3, color='#4a5568')

# Add value labels on bars
for bars, vals in [(bars1, [call_buys, put_buys]), (bars2, [call_sells, put_sells])]:
    for bar, val in zip(bars, vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ax3.get_ylim()[1] * 0.01,
                 f'{val:,}', ha='center', va='bottom', color='white', fontsize=10)

# Panel 4: Premium by direction
ax4 = axes[1, 1]
ax4.set_facecolor("#0a1628" if DARK_THEME else "white")

call_buy_prem = df[(df['opt_type'] == 'C') & (df['trade_dir'] == 'BUY')]['premium'].sum()
call_sell_prem = df[(df['opt_type'] == 'C') & (df['trade_dir'] == 'SELL')]['premium'].sum()
put_buy_prem = df[(df['opt_type'] == 'P') & (df['trade_dir'] == 'BUY')]['premium'].sum()
put_sell_prem = df[(df['opt_type'] == 'P') & (df['trade_dir'] == 'SELL')]['premium'].sum()

bars1 = ax4.bar(x - width/2, [call_buy_prem/1e6, put_buy_prem/1e6], width, 
                label='BUY Premium', color='#22c55e', alpha=0.85)
bars2 = ax4.bar(x + width/2, [call_sell_prem/1e6, put_sell_prem/1e6], width, 
                label='SELL Premium', color='#ef4444', alpha=0.85)

ax4.set_xticks(x)
ax4.set_xticklabels(['CALLS', 'PUTS'], color='white', fontsize=14)
ax4.set_ylabel('Premium ($M)', fontsize=12, color='white')
total_prem = (call_buy_prem + call_sell_prem + put_buy_prem + put_sell_prem) / 1e6
ax4.set_title(f'Premium by Option Type & Direction\n(Total: ${total_prem:.1f}M)', 
              color='white', fontsize=14, fontweight='bold')
ax4.legend(loc='upper right', framealpha=0.8, facecolor='#1a2744', fontsize=12)
ax4.tick_params(colors='white')
for spine in ax4.spines.values(): spine.set_color('#4a5568')
ax4.grid(axis='y', alpha=0.3, color='#4a5568')

# Add premium labels on bars
for bars, vals in [(bars1, [call_buy_prem, put_buy_prem]), (bars2, [call_sell_prem, put_sell_prem])]:
    for bar, val in zip(bars, vals):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + ax4.get_ylim()[1] * 0.01,
                 f'${val/1e6:.1f}M', ha='center', va='bottom', color='white', fontsize=10)

plt.suptitle(f"Trade Distribution Analysis — {TRADE_DATE} | {TIME_LABEL}", 
             color='white', fontsize=16, fontweight='bold', y=1.02)
fig.text(0.99, 0.01, f"Generated: {GENERATED_AT}", ha="right", va="bottom", 
         fontsize=8, color="gray", transform=fig.transFigure)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SUMMARY
# ============================================================

print(f"""
{'='*70}
0DTE ANALYSIS SUMMARY — {TRADE_DATE} | {TIME_LABEL}
{'='*70}

DATA:
  Trades: {len(df):,} | Contracts: {df['size'].sum():,} | Spot: ${result.spot:,.0f}

GEX ANALYSIS:
  Traditional GEX:   ${result.trad_net/1e6:>8.2f}M  {'STABILIZING' if result.trad_net > 0 else 'DESTABILIZING'}
  Side-Weighted GEX: ${result.sw_net/1e6:>8.2f}M  {'STABILIZING' if result.sw_net > 0 else 'DESTABILIZING'}

TRADE CLASSIFICATION:
  Buy Rates: Calls {result.call_buy_pct:.0f}% | Puts {result.put_buy_pct:.0f}%
  mid_market classified as: BUY
  
SIDE BREAKDOWN:
{df['side'].value_counts().to_string()}

{'='*70}
Generated: {GENERATED_AT}
{'='*70}
""")